# Fine-tuning Llama-3.2-3B-Instruct on Qasper

Run setup:
- Model: `meta-llama/Llama-3.2-3B-Instruct`.
- Runtime: Kaggle with **2 Tesla T4** GPUs.
- Training: 4-bit QLoRA with gradient checkpointing and mixed precision.
- Context packing prioritizes the annotated evidence fields; the EDA reports 87% coverage.
- Per-device batch size: 4; gradient accumulation: 4.
- Dataset path: `/kaggle/input/datasets/yassireyassir/qasper/data`.
- Evaluation files: `metrics.json` and `evaluation.json`.

## 1) Configuration

The main run settings are grouped in this cell.

In [1]:
from pathlib import Path
import math

# -----------------------------
# Runtime profile
# -----------------------------
# Settings used for the Kaggle 2x T4 run.
RUNTIME_PROFILE = "kaggle_fast"  # options: kaggle_fast, full_quality

if RUNTIME_PROFILE == "kaggle_fast":
    EPOCHS = 3
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 16
    LEARNING_RATE = 1e-4
    MAX_SEQ_LENGTH = 2048
    OOM_SEQ_LENGTH_FALLBACKS = [2048, 1536, 1024]

    # Leave room for the full cleaned training split.
    MAX_TRAIN_SAMPLES = 2000
    MAX_VAL_SAMPLES = 800
    MAX_TEST_SAMPLES = 1200

    ENABLE_CLASS_BALANCE_OVERSAMPLING = True
    OVER_SAMPLE_TYPES = {"hybrid", "boolean"}

    # Optional train/validation resplit; the test split stays unchanged.
    USE_TRAIN_VAL_RESPLIT = False
    TRAIN_VAL_RESPLIT_RATIO = 0.92
    TRAIN_VAL_RESPLIT_STRATIFY = True
    TRAIN_VAL_RESPLIT_DROP_DUP_CROSS_SPLIT = True

    # Generation limits used during evaluation.
    EVAL_MAX_SAMPLES_VALIDATION = 600
    EVAL_MAX_SAMPLES_TEST = 800
    MAX_NEW_TOKENS_EVAL = 96
    GEN_BATCH_SIZE = 1
else:
    EPOCHS = 3
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 16
    LEARNING_RATE = 5e-5
    MAX_SEQ_LENGTH = 3072
    OOM_SEQ_LENGTH_FALLBACKS = [3072, 2560, 2048, 1792, 1536, 1280, 1024, 896, 768, 640, 512]

    MAX_TRAIN_SAMPLES = None
    MAX_VAL_SAMPLES = None
    MAX_TEST_SAMPLES = None

    ENABLE_CLASS_BALANCE_OVERSAMPLING = True
    OVER_SAMPLE_TYPES = {"hybrid", "boolean"}

    USE_TRAIN_VAL_RESPLIT = False
    TRAIN_VAL_RESPLIT_RATIO = 0.90
    TRAIN_VAL_RESPLIT_STRATIFY = True
    TRAIN_VAL_RESPLIT_DROP_DUP_CROSS_SPLIT = True

    EVAL_MAX_SAMPLES_VALIDATION = None
    EVAL_MAX_SAMPLES_TEST = None
    MAX_NEW_TOKENS_EVAL = 128
    GEN_BATCH_SIZE = 1

# Alias retained for older cells.
EVAL_MAX_SAMPLES = EVAL_MAX_SAMPLES_VALIDATION

# These values are adjusted after GPU detection.
TRAIN_BSZ = BATCH_SIZE
EVAL_BSZ = 1
GRAD_ACC_STEPS = GRADIENT_ACCUMULATION_STEPS

# LoRA defaults for Llama-3.2-3B-Instruct on Kaggle 2xT4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# -----------------------------
# Core model and data settings
# -----------------------------
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/yassirchergui/qasper/data")
LOCAL_DATA_DIR = Path("data")

# EDA + plan references
KAGGLE_EDA_JSON = KAGGLE_DATA_DIR / "eda.json"
LOCAL_EDA_JSON = LOCAL_DATA_DIR / "eda.json"
IMPLEMENTATION_MD_PATH = Path("implementation.md")

# Preprocessing and packing
CHUNK_SIZE = 384
CHUNK_STRIDE = 96
# Reserve about 200 tokens for the system prompt, question, and answer prefix.
MAX_CONTEXT_TOKENS_IN_PROMPT = max(384, min(2048, MAX_SEQ_LENGTH - 200))
# Evidence coverage and length estimates come from the EDA.
USE_EVIDENCE_FOR_PACKING = True
EVIDENCE_BOOST_WEIGHT = 0.4  # weight for evidence overlap in chunk scoring
EVAL_CONTEXT_RECALL = True   # measure evidence recall in packed context

MIN_QUESTION_WORDS = 2
MAX_QUESTION_WORDS = 64
MAX_ANSWER_WORDS = 256

# Training stability and speed
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 0.3
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 5
EVAL_STEPS = 250
SAVE_STEPS = 250
SAVE_TOTAL_LIMIT = 2

# Trainer scheduling behavior
TRAIN_EVAL_STRATEGY = "epoch"   # "no", "steps", "epoch"
TRAIN_SAVE_STRATEGY = "epoch"   # "steps", "epoch"
LOAD_BEST_MODEL_AT_END = True
GROUP_BY_LENGTH = True

# Confidence and metric runtime knobs
CONFIDENCE_MODE = "auto"  # "auto", "logprob", "proxy"
CONFIDENCE_QUANTILE = 0.75
ENABLE_TOKEN_LOGPROB_CONFIDENCE = True
CONFIDENCE_PROXY_FLOOR = 0.05
ENABLE_BERTSCORE = True
BERTSCORE_MAX_SAMPLES = 256
BERTSCORE_DEVICE = "cpu"
BERTSCORE_MODEL = "distilbert-base-uncased"

# Runtime behavior
INSTALL_DEPENDENCIES = True
DEPENDENCY_INSTALL_MODE = "if-missing"  # "if-missing" or "upgrade"
ENABLE_CACHE = True
SEED = 42

# Output
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR = WORK_DIR / "qasper_llama_3_1_8b_qlora"
CACHE_DIR = OUTPUT_DIR / "cache"
PRED_DIR = OUTPUT_DIR / "predictions"

for p in [OUTPUT_DIR, CACHE_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
expected_updates = math.ceil(MAX_TRAIN_SAMPLES / effective_batch) if MAX_TRAIN_SAMPLES else None

print("Configuration ready.")
print("RUNTIME_PROFILE:", RUNTIME_PROFILE)
print("MODEL_ID:", MODEL_ID)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("MAX_CONTEXT_TOKENS_IN_PROMPT:", MAX_CONTEXT_TOKENS_IN_PROMPT)
print("OOM fallback lengths:", OOM_SEQ_LENGTH_FALLBACKS)
print("LoRA (r, alpha, dropout):", (LORA_R, LORA_ALPHA, LORA_DROPOUT))
print("LoRA target modules:", LORA_TARGET_MODULES)
print("Sample caps train/val/test:", (MAX_TRAIN_SAMPLES, MAX_VAL_SAMPLES, MAX_TEST_SAMPLES))
print("Eval sample caps val/test:", (EVAL_MAX_SAMPLES_VALIDATION, EVAL_MAX_SAMPLES_TEST))
print("Trainer eval strategy:", TRAIN_EVAL_STRATEGY)
print("Load best model at end:", LOAD_BEST_MODEL_AT_END)
print("Use train+val resplit:", USE_TRAIN_VAL_RESPLIT)
print("Confidence mode:", CONFIDENCE_MODE)
print("ENABLE_TOKEN_LOGPROB_CONFIDENCE:", ENABLE_TOKEN_LOGPROB_CONFIDENCE)
print("KAGGLE_DATA_DIR:", KAGGLE_DATA_DIR)
if expected_updates is not None:
    print("Approx optimizer updates per epoch:", expected_updates)

Configuration ready.
RUNTIME_PROFILE: kaggle_fast
MODEL_ID: meta-llama/Llama-3.2-3B-Instruct
MAX_SEQ_LENGTH: 2048
MAX_CONTEXT_TOKENS_IN_PROMPT: 1848
OOM fallback lengths: [2048, 1536, 1024]
LoRA (r, alpha, dropout): (16, 32, 0.05)
LoRA target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
Sample caps train/val/test: (2000, 800, 1200)
Eval sample caps val/test: (600, 800)
Trainer eval strategy: epoch
Load best model at end: True
Use train+val resplit: False
Confidence mode: auto
ENABLE_TOKEN_LOGPROB_CONFIDENCE: True
KAGGLE_DATA_DIR: /kaggle/input/datasets/yassirchergui/qasper/data
Approx optimizer updates per epoch: 125


## 2) Environment and imports

This cell installs and imports the dependencies used by the Trainer workflow.

In [2]:
import os
import sys
import subprocess
import importlib
import importlib.util
from importlib.metadata import PackageNotFoundError, version as pkg_version

try:
    from packaging.version import Version
except Exception:
    _ = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "packaging"], check=True)
    from packaging.version import Version

REQUIRED_PACKAGE_SPECS = {
    # Keep a recent stack for Mistral-7B-Instruct-v0.3 and Trainer compatibility.
    "transformers": "transformers>=4.45.0",
    "datasets": "datasets>=2.20.0",
    "accelerate": "accelerate>=0.33.0",
    "peft": "peft>=0.11.1",
    "bitsandbytes": "bitsandbytes>=0.43.1",
    "evaluate": "evaluate>=0.4.2",
    "bert-score": "bert-score>=0.3.13",
    "rouge-score": "rouge-score>=0.1.2",
    "sacrebleu": "sacrebleu>=2.4.2",
    "scikit-learn": "scikit-learn>=1.3.0",
    "pandas": "pandas>=2.0.0",
    "pyarrow": "pyarrow>=16.1.0",
    "matplotlib": "matplotlib>=3.7.0",
    "seaborn": "seaborn>=0.13.0",
}

PACKAGE_IMPORT_NAME = {
    "scikit-learn": "sklearn",
    "bert-score": "bert_score",
    "rouge-score": "rouge_score",
}


def module_exists(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def parse_min_version(spec: str) -> str:
    if ">=" in spec:
        return spec.split(">=", 1)[1].strip()
    raise ValueError(f"Unsupported spec format (expected >=): {spec}")


def is_outdated(dist_name: str, min_required: str) -> bool:
    try:
        installed = pkg_version(dist_name)
    except PackageNotFoundError:
        return True
    return Version(installed) < Version(min_required)


def get_loaded_module_version(module_name: str):
    mod = sys.modules.get(module_name)
    if mod is None:
        return None
    return getattr(mod, "__version__", None)


def clear_module_cache(module_name: str) -> int:
    keys = [k for k in list(sys.modules.keys()) if k == module_name or k.startswith(f"{module_name}.")]
    for k in keys:
        del sys.modules[k]
    return len(keys)


if INSTALL_DEPENDENCIES:
    package_specs = list(REQUIRED_PACKAGE_SPECS.values())

    if DEPENDENCY_INSTALL_MODE == "upgrade":
        to_install = package_specs
    else:
        to_install = []
        for pkg_name, spec in REQUIRED_PACKAGE_SPECS.items():
            import_name = PACKAGE_IMPORT_NAME.get(pkg_name, pkg_name.replace("-", "_"))
            min_required = parse_min_version(spec)

            if not module_exists(import_name):
                to_install.append(spec)
                continue

            if is_outdated(pkg_name, min_required):
                to_install.append(spec)

    if to_install:
        print(f"Installing/upgrading {len(to_install)} package(s) to satisfy minimum versions...")
        _ = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + to_install, check=True)
        print("Dependency installation complete.")
    else:
        print("All required packages satisfy minimum versions. Skipping pip install.")

    # If transformers was already imported before upgrade, clear stale in-memory modules.
    tf_required = parse_min_version(REQUIRED_PACKAGE_SPECS["transformers"])
    tf_loaded = get_loaded_module_version("transformers")
    tf_installed = pkg_version("transformers")

    stale_loaded = False
    if tf_loaded is not None:
        try:
            stale_loaded = (Version(tf_loaded) < Version(tf_required)) or (Version(tf_loaded) < Version(tf_installed))
        except Exception:
            stale_loaded = True

    if stale_loaded:
        cleared = 0
        for mod_name in ["transformers", "peft", "accelerate", "trl"]:
            cleared += clear_module_cache(mod_name)
        importlib.invalidate_caches()
        print(
            "Detected stale in-memory ML modules. "
            f"Cleared {cleared} cached modules to force fresh imports (loaded={tf_loaded}, installed={tf_installed})."
        )
else:
    print("Skipping dependency installation.")

Installing/upgrading 4 package(s) to satisfy minimum versions...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.2 MB/s eta 0:00:00
Dependency installation complete.


In [3]:
from huggingface_hub import login

# Authenticate to Hugging Face to access the gated Mistral model
login()


In [4]:
import gc
import json
import math
import random
import re
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from contextlib import nullcontext

import evaluate
import numpy as np
import pandas as pd
import torch
import seaborn as sns

from datasets import Dataset
from sklearn.metrics import brier_score_loss, roc_auc_score

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="talk")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        print(f"GPU {idx}: {torch.cuda.get_device_name(idx)}")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3) Load the Qasper splits and EDA settings

The loader checks the Kaggle dataset directory first and uses the local path as a fallback.

In [5]:
def resolve_data_root() -> Path:
    if KAGGLE_DATA_DIR.exists():
        return KAGGLE_DATA_DIR
    if LOCAL_DATA_DIR.exists():
        return LOCAL_DATA_DIR
    raise FileNotFoundError(
        "Could not find data directory. Expected /kaggle/input/datasets/yassireyassir/qasper/data or ./data"
    )


def resolve_eda_path(data_root: Path) -> Path:
    candidates = [
        KAGGLE_EDA_JSON,
        LOCAL_EDA_JSON,
        data_root / "eda.json",
        Path("eda.json"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find eda.json")


def resolve_csv_dir(data_root: Path) -> Path:
    csv_dir = data_root / "csv"
    return csv_dir if csv_dir.exists() else data_root


def load_split(split_name: str, csv_dir: Path) -> pd.DataFrame:
    fp = csv_dir / f"{split_name}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Missing split file: {fp}")
    df = pd.read_csv(fp)
    df["split"] = split_name
    return df


def normalize_split_label(value: Any) -> str:
    txt = str(value).strip().lower()
    if txt.startswith("val"):
        return "validation"
    if txt.startswith("test"):
        return "test"
    return "train"


def derive_strata_label(df: pd.DataFrame) -> pd.Series:
    ans_type = df.get("answer_type_initial", pd.Series([""] * len(df), index=df.index)).astype(str).str.strip().str.lower()
    is_unans = df.get("is_unanswerable", pd.Series([False] * len(df), index=df.index))
    is_unans = is_unans.fillna(False).map(lambda x: str(x).strip().lower() in {"1", "true", "yes", "y", "t"})
    ans_type = np.where(is_unans, "unanswerable", ans_type)
    ans_type = pd.Series(ans_type, index=df.index).replace({"": "unknown", "nan": "unknown", "none": "unknown"})
    return ans_type.astype(str)


def stratified_resplit_train_validation(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    train_ratio: float,
    stratify: bool,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    combined = pd.concat([train_df.copy(), val_df.copy()], ignore_index=True)
    before_rows = len(combined)

    dedup_removed = 0
    if TRAIN_VAL_RESPLIT_DROP_DUP_CROSS_SPLIT:
        dedup_subset = [c for c in ["paper_id", "question_uid", "answer_text"] if c in combined.columns]
        if dedup_subset:
            combined = combined.drop_duplicates(subset=dedup_subset, keep="first").reset_index(drop=True)
            dedup_removed = int(before_rows - len(combined))

    if len(combined) < 2:
        return train_df.copy(), val_df.copy(), {
            "applied": False,
            "reason": "combined_train_validation_too_small",
            "combined_rows": int(len(combined)),
        }

    ratio = float(np.clip(train_ratio, 0.5, 0.98))
    rng = np.random.default_rng(SEED)

    combined = combined.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    if stratify:
        combined["_strata"] = derive_strata_label(combined)
        selected_train_idx: List[int] = []

        for _, grp in combined.groupby("_strata", sort=False):
            grp_idx = grp.index.to_numpy()
            if len(grp_idx) <= 1:
                selected_train_idx.extend(grp_idx.tolist())
                continue

            n_train = int(round(len(grp_idx) * ratio))
            n_train = min(max(1, n_train), len(grp_idx) - 1)
            chosen = rng.choice(grp_idx, size=n_train, replace=False)
            selected_train_idx.extend(chosen.tolist())

        train_mask = combined.index.isin(selected_train_idx)
    else:
        n_train = int(round(len(combined) * ratio))
        n_train = min(max(1, n_train), len(combined) - 1)
        train_mask = np.zeros(len(combined), dtype=bool)
        train_mask[:n_train] = True

    train_new = combined.loc[train_mask].copy().reset_index(drop=True)
    val_new = combined.loc[~train_mask].copy().reset_index(drop=True)

    if "_strata" in train_new.columns:
        train_new = train_new.drop(columns=["_strata"])
    if "_strata" in val_new.columns:
        val_new = val_new.drop(columns=["_strata"])

    # Keep a non-empty validation split.
    if len(val_new) == 0 and len(train_new) > 1:
        val_new = train_new.tail(1).copy().reset_index(drop=True)
        train_new = train_new.iloc[:-1].copy().reset_index(drop=True)

    train_new["split"] = "train"
    val_new["split"] = "validation"

    stats = {
        "applied": True,
        "combined_rows_before": int(before_rows),
        "combined_rows_after": int(len(combined)),
        "dedup_removed_cross_split": int(dedup_removed),
        "train_rows": int(len(train_new)),
        "validation_rows": int(len(val_new)),
        "train_ratio_requested": float(ratio),
        "stratified": bool(stratify),
    }
    return train_new, val_new, stats


data_root = resolve_data_root()
csv_dir = resolve_csv_dir(data_root)
eda_path = resolve_eda_path(data_root)

with eda_path.open("r", encoding="utf-8") as f:
    eda = json.load(f)

plan_excerpt = ""
if IMPLEMENTATION_MD_PATH.exists():
    plan_excerpt = IMPLEMENTATION_MD_PATH.read_text(encoding="utf-8")[:1200]

train_df_raw = load_split("train", csv_dir)
val_df_raw = load_split("validation", csv_dir)
test_df_raw = load_split("test", csv_dir)

required_columns = [
    "split",
    "paper_id",
    "question_id",
    "question_uid",
    "answer_id",
    "question_text",
    "context_text",
    "answer_text",
    "answer_type_initial",
    "is_unanswerable",
]
for split_name, split_df in [
    ("train", train_df_raw),
    ("validation", val_df_raw),
    ("test", test_df_raw),
]:
    missing_cols = [c for c in required_columns if c not in split_df.columns]
    if missing_cols:
        raise ValueError(f"{split_name} split missing expected columns: {missing_cols}")

resplit_stats = {"applied": False}
if USE_TRAIN_VAL_RESPLIT:
    train_df_raw, val_df_raw, resplit_stats = stratified_resplit_train_validation(
        train_df_raw,
        val_df_raw,
        train_ratio=TRAIN_VAL_RESPLIT_RATIO,
        stratify=TRAIN_VAL_RESPLIT_STRATIFY,
    )

for split_df, split_name in [
    (train_df_raw, "train"),
    (val_df_raw, "validation"),
    (test_df_raw, "test"),
]:
    split_df["split"] = split_name

print("Data root:", data_root)
print("CSV dir:", csv_dir)
print("EDA path:", eda_path)
print("Rows -> train / val / test:", len(train_df_raw), len(val_df_raw), len(test_df_raw))
print("Plan available:", IMPLEMENTATION_MD_PATH.exists())
print("Train+validation resplit:", json.dumps(resplit_stats, indent=2))

eda_context_budget = int(eda.get("key_insights", {}).get("typical_context_size_tokens", {}).get("recommended_budget", 4096))
eda_answer_budget = int(eda.get("key_insights", {}).get("answer_style_variability", {}).get("recommended_answer_budget", 75))
eda_hard_ratio = float(eda.get("key_insights", {}).get("difficulty_estimation", {}).get("estimated_hard_sample_ratio", 0.70))

# Read the evidence summary produced by the EDA.
eda_evidence = eda.get("key_insights", {}).get("evidence_analysis", {})
eda_evidence_coverage = float(eda_evidence.get("evidence_coverage_pct", 0.0))
eda_highlighted_coverage = float(eda_evidence.get("highlighted_evidence_coverage_pct", 0.0))
eda_avg_evidence_paras = float(eda_evidence.get("avg_evidence_paragraphs", 0.0))

# Detect the optional evidence columns.
evidence_columns_available = all(
    col in train_df_raw.columns
    for col in ["evidence_text", "highlighted_evidence_text", "has_evidence"]
)

print("EDA recommended context budget:", eda_context_budget)
print("EDA recommended answer budget:", eda_answer_budget)
print("EDA hard-sample ratio:", eda_hard_ratio)
print("EDA evidence coverage:", f"{eda_evidence_coverage}%")
print("Evidence columns available in data:", evidence_columns_available)
if evidence_columns_available:
    ev_non_empty = (train_df_raw["evidence_text"].fillna("").astype(str).str.len() > 0).sum()
    print(f"  Train evidence non-empty: {ev_non_empty}/{len(train_df_raw)}")

display(train_df_raw.head(2))

Data root: /kaggle/input/datasets/yassirchergui/qasper/data
CSV dir: /kaggle/input/datasets/yassirchergui/qasper/data/csv
EDA path: /kaggle/input/datasets/yassirchergui/qasper/data/eda.json
Rows -> train / val / test: 2675 1764 3554
Plan available: False
Train+validation resplit: {
  "applied": false
}
EDA recommended context budget: 4096
EDA recommended answer budget: 75
EDA hard-sample ratio: 0.700613036406856
EDA evidence coverage: 87.2%
Evidence columns available in data: True
  Train evidence non-empty: 2308/2675


,split,paper_id,question_id,question_uid,answer_id,question_text,context_text,annotation_id,worker_id,answer_text,answer_type_initial,yes_no,is_unanswerable,extractive_spans,evidence_text,evidence_paragraphs,highlighted_evidence_text,highlighted_evidence_spans,has_evidence,has_highlighted_evidence,num_evidence_paragraphs,num_highlighted_sentences,num_extractive_spans,answer_in_context,answer_in_evidence,answer_in_highlighted
0,train,1909.00694,753990d0b621d390ed58f20c4d9e4f065f0dc672,1909.00694::753990d0b621d390ed58f20c4d9e4f065f...,1909.00694::753990d0b621d390ed58f20c4d9e4f065f...,What is the seed lexicon?,Minimally Supervised Learning of Affective Eve...,31e85022a847f37c15fd0415f3c450c74c8e4755,c1fbdd7a261021041f75fbe00a55b4c386ebbbb4,a vocabulary of positive and negative predicat...,abstractive,NaN,False,[],The seed lexicon consists of positive and nega...,['The seed lexicon consists of positive and ne...,The seed lexicon consists of positive and nega...,['The seed lexicon consists of positive and ne...,True,True,1,2,0,False,False,False
1,train,1909.00694,753990d0b621d390ed58f20c4d9e4f065f0dc672,1909.00694::753990d0b621d390ed58f20c4d9e4f065f...,1909.00694::753990d0b621d390ed58f20c4d9e4f065f...,What is the seed lexicon?,Minimally Supervised Learning of Affective Eve...,95da0a6e1b08db74a405c6a71067c9b272a50ff5,2cfd959e433f290bb50b55722370f0d22fe090b7,seed lexicon consists of positive and negative...,extractive,NaN,False,['seed lexicon consists of positive and negati...,The seed lexicon consists of positive and nega...,['The seed lexicon consists of positive and ne...,The seed lexicon consists of positive and nega...,['The seed lexicon consists of positive and ne...,True,True,1,1,1,True,True,True


## 4) Preprocess and pack contexts

The preprocessing step normalizes text, removes unusable or duplicate rows, and packs long contexts to the configured prompt budget.

In [6]:
print("Loading tokenizer for chunking and training prompt construction...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

WORD_RE = re.compile(r"\b\w+\b", re.UNICODE)


def safe_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)


def normalize_text(text: Any) -> str:
    text = safe_text(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u0000-\u001F\u007F]", " ", text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def norm_key(text: Any) -> str:
    return re.sub(r"\s+", " ", normalize_text(text).lower()).strip()


def word_count(text: str) -> int:
    return len(WORD_RE.findall(normalize_text(text).lower()))


def simple_tokens(text: str) -> set:
    return set(WORD_RE.findall(normalize_text(text).lower()))


def parse_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, np.integer)):
        return bool(value)
    txt = norm_key(value)
    if txt in {"1", "true", "yes", "y", "t"}:
        return True
    if txt in {"0", "false", "no", "n", "f", ""}:
        return False
    return bool(value)


SUSPICIOUS_UNICODE_CHARS = {chr(0xFFFD), chr(0x200B), chr(0x2060)}


def has_suspicious_unicode(text: Any) -> bool:
    txt = safe_text(text)
    return any(ch in txt for ch in SUSPICIOUS_UNICODE_CHARS)


def resolve_answer_type(row: pd.Series) -> str:
    base_type = norm_key(row.get("answer_type", ""))
    if not base_type:
        base_type = norm_key(row.get("answer_type_initial", ""))

    is_unanswerable = parse_bool(row.get("is_unanswerable", False))
    if is_unanswerable:
        return "unanswerable"

    if base_type in {"extractive", "abstractive", "hybrid", "unanswerable", "other", "boolean"}:
        return base_type

    return "abstractive"


def clean_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int]]:
    d = df.copy()
    before = len(d)

    # 1. Resolve answer types immediately
    d["answer_type"] = d.apply(resolve_answer_type, axis=1)

    # 2. Strictly drop "unanswerable" questions before any other processing
    d = d[d["answer_type"] != "unanswerable"]

    # 3. Proceed with standard processing
    text_cols = [
        "question_text",
        "context_text",
        "answer_text",
        "answer_type_initial",
        "paper_id",
        "question_uid",
        "answer_id",
        "evidence_text",
        "highlighted_evidence_text",
    ]
    for col in text_cols:
        if col in d.columns:
            d[col] = d[col].map(normalize_text)

    d["question_words"] = d["question_text"].map(word_count)
    d["answer_words"] = d["answer_text"].map(word_count)

    d = d[d["question_text"].str.len() > 0]

    empty_non_unans = (d["answer_words"] == 0) & (d["answer_type"] != "unanswerable")
    d = d[~empty_non_unans]

    other_mask = d["answer_type"] == "other"
    other_empty = other_mask & (d["answer_words"] == 0)
    other_non_empty = other_mask & (d["answer_words"] > 0)
    d.loc[other_empty, "answer_type"] = "unanswerable"
    d.loc[other_non_empty, "answer_type"] = "abstractive"

    unans_mask = d["answer_type"] == "unanswerable"
    d.loc[unans_mask, "answer_text"] = "INSUFFICIENT_CONTEXT"
    d = d[d["answer_type"] != "unanswerable"]

    d = d[(d["question_words"] >= MIN_QUESTION_WORDS) & (d["question_words"] <= MAX_QUESTION_WORDS)]
    d = d[d["answer_words"] <= MAX_ANSWER_WORDS]

    suspicious = d["context_text"].map(has_suspicious_unicode)
    d = d[~suspicious]

    d["norm_question"] = d["question_text"].map(norm_key)
    d["norm_answer"] = d["answer_text"].map(norm_key)
    pre_dedup = len(d)
    d = d.drop_duplicates(subset=["split", "paper_id", "norm_question", "norm_answer"], keep="first")

    stats = {
        "rows_before": int(before),
        "rows_after": int(len(d)),
        "removed_total": int(before - len(d)),
        "removed_by_dedup": int(pre_dedup - len(d)),
    }

    d = d.reset_index(drop=True)
    return d, stats


def stratified_cap(df: pd.DataFrame, max_rows: Optional[int], split_name: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if max_rows is None or len(df) <= max_rows:
        return df.reset_index(drop=True), {
            "split": split_name,
            "original_rows": int(len(df)),
            "kept_rows": int(len(df)),
            "cap": max_rows,
            "applied": False,
        }

    rng = np.random.default_rng(SEED)
    d = df.copy().reset_index(drop=True)
    d["_grp"] = d["answer_type"].fillna("unknown").astype(str)

    groups = {k: v.index.to_numpy() for k, v in d.groupby("_grp", sort=False)}
    n_groups = max(1, len(groups))
    base_take = max(1, max_rows // n_groups)

    selected_idx: List[int] = []
    leftovers: List[int] = []

    for _, idxs in groups.items():
        idxs = np.asarray(idxs)
        if len(idxs) <= base_take:
            selected_idx.extend(idxs.tolist())
        else:
            chosen = rng.choice(idxs, size=base_take, replace=False)
            selected_idx.extend(chosen.tolist())
            remain = np.setdiff1d(idxs, chosen, assume_unique=False)
            leftovers.extend(remain.tolist())

    if len(selected_idx) < max_rows and len(leftovers) > 0:
        needed = min(max_rows - len(selected_idx), len(leftovers))
        extra = rng.choice(np.asarray(leftovers), size=needed, replace=False)
        selected_idx.extend(extra.tolist())

    selected_idx = sorted(set(selected_idx))
    if len(selected_idx) > max_rows:
        selected_idx = sorted(rng.choice(np.asarray(selected_idx), size=max_rows, replace=False).tolist())

    out = d.iloc[selected_idx].copy().drop(columns=["_grp"]).reset_index(drop=True)
    stats = {
        "split": split_name,
        "original_rows": int(len(df)),
        "kept_rows": int(len(out)),
        "cap": int(max_rows),
        "applied": True,
    }
    return out, stats


def split_paragraphs(text: str) -> List[str]:
    chunks = [normalize_text(x) for x in re.split(r"\n{2,}|\r\n{2,}", safe_text(text))]
    chunks = [c for c in chunks if c]
    if not chunks:
        return [normalize_text(text)] if normalize_text(text) else []
    return chunks


def token_chunk_text(text: str, chunk_size: int, stride: int) -> List[str]:
    text = normalize_text(text)
    if not text:
        return []

    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= chunk_size:
        return [text]

    step = max(1, chunk_size - stride)
    out = []
    start = 0
    while start < len(ids):
        end = min(start + chunk_size, len(ids))
        sub_ids = ids[start:end]
        chunk = tokenizer.decode(sub_ids, skip_special_tokens=True).strip()
        if chunk:
            out.append(chunk)
        if end >= len(ids):
            break
        start += step
    return out


_context_chunks_cache: Dict[Tuple[int, int], List[str]] = {}


def build_context_chunks(context: str) -> List[str]:
    ctx = normalize_text(context)
    key = (len(ctx), hash(ctx))
    if key in _context_chunks_cache:
        return _context_chunks_cache[key]

    paras = split_paragraphs(ctx)
    chunks: List[str] = []
    for p in paras:
        if not p:
            continue
        chunks.extend(token_chunk_text(p, CHUNK_SIZE, CHUNK_STRIDE))
    if not chunks:
        chunks = token_chunk_text(ctx, CHUNK_SIZE, CHUNK_STRIDE)

    chunks = chunks[:96]
    _context_chunks_cache[key] = chunks
    return chunks


def score_chunk(
    question_tokens: set,
    chunk_tokens: set,
    idx: int,
    evidence_tokens: Optional[set] = None,
    highlighted_tokens: Optional[set] = None,
) -> float:
    """Score a chunk for selection. When evidence is available, boost chunks
    that overlap with evidence paragraphs / highlighted evidence sentences.
    EDA insight: evidence median=146 tokens, highlighted median=57 tokens."""
    if not chunk_tokens:
        return -1.0
    q_overlap = len(question_tokens & chunk_tokens) / max(1, len(question_tokens))
    lead_bonus = 0.04 / (1.0 + idx)
    base_score = 0.96 * q_overlap + lead_bonus

    if not USE_EVIDENCE_FOR_PACKING:
        return base_score

    # Give evidence passages a higher packing score.
    ev_score = 0.0
    if evidence_tokens and len(evidence_tokens) > 0:
        ev_score = len(evidence_tokens & chunk_tokens) / max(1, len(evidence_tokens))
    hl_score = 0.0
    if highlighted_tokens and len(highlighted_tokens) > 0:
        hl_score = len(highlighted_tokens & chunk_tokens) / max(1, len(highlighted_tokens))

    # Sentence-level highlights receive the larger evidence bonus.
    combined_ev = max(ev_score, hl_score * 1.2)
    return (1.0 - EVIDENCE_BOOST_WEIGHT) * base_score + EVIDENCE_BOOST_WEIGHT * combined_ev


def pack_context(
    question: str,
    context: str,
    token_budget: int,
    evidence_text: str = "",
    highlighted_evidence_text: str = "",
) -> Tuple[str, int, int, int]:
    c_text = normalize_text(context)
    if not c_text:
        return "", 0, 0, 0
        
    c_ids = tokenizer.encode(c_text, add_special_tokens=False)
    total_tokens = len(c_ids)
    
    if total_tokens <= token_budget:
        return c_text, 1, 1, total_tokens

    ev_text = normalize_text(evidence_text)
    start_tok = 0

    if ev_text:
        try:
            char_idx = c_text.index(ev_text)
            char_ratio = char_idx / len(c_text)
            ev_center_tok = int(total_tokens * char_ratio) + (len(tokenizer.encode(ev_text)) // 2)
            start_tok = max(0, ev_center_tok - (token_budget // 2))
        except ValueError:
            pass
            
    end_tok = min(total_tokens, start_tok + token_budget)
    if end_tok - start_tok < token_budget:
        start_tok = max(0, end_tok - token_budget)

    packed_ids = c_ids[start_tok:end_tok]
    packed = tokenizer.decode(packed_ids, skip_special_tokens=True).strip()
    return f"[CHUNK 1]\n{packed}", 1, 1, len(packed_ids)


def apply_context_packing(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = df.copy()

    packed_contexts = []
    used_chunk_counts = []
    candidate_chunk_counts = []
    packed_token_lengths = []

    pack_cache: Dict[Tuple, Tuple[str, int, int, int]] = {}

    # Detect the optional evidence columns.
    has_ev_col = "evidence_text" in out.columns
    has_hl_col = "highlighted_evidence_text" in out.columns

    for row in out.itertuples(index=False):
        q = normalize_text(row.question_text)
        c = normalize_text(row.context_text)
        ev = normalize_text(getattr(row, "evidence_text", "")) if has_ev_col else ""
        hl = normalize_text(getattr(row, "highlighted_evidence_text", "")) if has_hl_col else ""
        cache_key = (q, len(c), hash(c), len(ev), hash(ev))

        if cache_key not in pack_cache:
            pack_cache[cache_key] = pack_context(
                question=q,
                context=c,
                token_budget=MAX_CONTEXT_TOKENS_IN_PROMPT,
                evidence_text=ev,
                highlighted_evidence_text=hl,
            )

        packed, used_n, total_n, tok_n = pack_cache[cache_key]
        packed_contexts.append(packed)
        used_chunk_counts.append(used_n)
        candidate_chunk_counts.append(total_n)
        packed_token_lengths.append(tok_n)

    out["packed_context"] = packed_contexts
    out["packed_context_used_chunks"] = used_chunk_counts
    out["packed_context_candidate_chunks"] = candidate_chunk_counts
    out["packed_context_tokens"] = packed_token_lengths
    out["packed_context"] = out["packed_context"].fillna("")

    out = out[out["packed_context"].str.len() > 0].reset_index(drop=True)
    out["split"] = split_name

    # Measure how much annotated evidence remains after packing.
    if EVAL_CONTEXT_RECALL and has_ev_col:
        def _evidence_recall(row):
            ev = normalize_text(safe_text(row.get("evidence_text", "")))
            pc = normalize_text(safe_text(row.get("packed_context", "")))
            if not ev:
                return float("nan")
            ev_toks = simple_tokens(ev)
            pc_toks = simple_tokens(pc)
            if not ev_toks:
                return float("nan")
            return len(ev_toks & pc_toks) / len(ev_toks)
        out["packed_context_evidence_recall"] = out.apply(_evidence_recall, axis=1)
    else:
        out["packed_context_evidence_recall"] = float("nan")

    return out


cache_tag = (
    f"{RUNTIME_PROFILE}_"
    f"s{MAX_SEQ_LENGTH}_ctx{MAX_CONTEXT_TOKENS_IN_PROMPT}_"
    f"tr{MAX_TRAIN_SAMPLES if MAX_TRAIN_SAMPLES is not None else 'all'}_"
    f"va{MAX_VAL_SAMPLES if MAX_VAL_SAMPLES is not None else 'all'}_"
    f"te{MAX_TEST_SAMPLES if MAX_TEST_SAMPLES is not None else 'all'}_v2"
)

train_cache = CACHE_DIR / f"train_preprocessed_{cache_tag}.parquet"
val_cache = CACHE_DIR / f"validation_preprocessed_{cache_tag}.parquet"
test_cache = CACHE_DIR / f"test_preprocessed_{cache_tag}.parquet"

if ENABLE_CACHE and train_cache.exists() and val_cache.exists() and test_cache.exists():
    train_df = pd.read_parquet(train_cache)
    val_df = pd.read_parquet(val_cache)
    test_df = pd.read_parquet(test_cache)
    clean_stats = {"cache": "loaded", "cache_tag": cache_tag}
else:
    train_clean, train_stats = clean_split(train_df_raw)
    val_clean, val_stats = clean_split(val_df_raw)
    test_clean, test_stats = clean_split(test_df_raw)

    train_clean, train_cap_stats = stratified_cap(train_clean, MAX_TRAIN_SAMPLES, "train")
    val_clean, val_cap_stats = stratified_cap(val_clean, MAX_VAL_SAMPLES, "validation")
    test_clean, test_cap_stats = stratified_cap(test_clean, MAX_TEST_SAMPLES, "test")

    train_df = apply_context_packing(train_clean, "train")
    val_df = apply_context_packing(val_clean, "validation")
    test_df = apply_context_packing(test_clean, "test")

    clean_stats = {
        "train": train_stats,
        "validation": val_stats,
        "test": test_stats,
        "caps": {
            "train": train_cap_stats,
            "validation": val_cap_stats,
            "test": test_cap_stats,
        },
        "cache_tag": cache_tag,
    }

    if ENABLE_CACHE:
        train_df.to_parquet(train_cache, index=False)
        val_df.to_parquet(val_cache, index=False)
        test_df.to_parquet(test_cache, index=False)

print("Preprocessing stats:")
print(json.dumps(clean_stats, indent=2))
print("Final rows -> train / val / test:", len(train_df), len(val_df), len(test_df))
print(
    "Mean packed context tokens -> train / val / test:",
    round(train_df["packed_context_tokens"].mean(), 2),
    round(val_df["packed_context_tokens"].mean(), 2),
    round(test_df["packed_context_tokens"].mean(), 2),
)

# Summarize the packed contexts.
if "packed_context_evidence_recall" in train_df.columns:
    ev_recall_valid = train_df["packed_context_evidence_recall"].dropna()
    if len(ev_recall_valid) > 0:
        print(f"Evidence recall in packed context (train): "
              f"mean={ev_recall_valid.mean():.3f}, "
              f"median={ev_recall_valid.median():.3f}, "
              f"p25={ev_recall_valid.quantile(0.25):.3f}, "
              f"p75={ev_recall_valid.quantile(0.75):.3f}")
        print(f"  Samples with evidence: {len(ev_recall_valid)}/{len(train_df)}")

display(train_df[["question_text", "answer_text", "answer_type", "packed_context_tokens", "packed_context_used_chunks"]].head(2))

Loading tokenizer for chunking and training prompt construction...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Preprocessing stats:
{
  "train": {
    "rows_before": 2675,
    "rows_after": 1973,
    "removed_total": 702,
    "removed_by_dedup": 8
  },
  "validation": {
    "rows_before": 1764,
    "rows_after": 1311,
    "removed_total": 453,
    "removed_by_dedup": 80
  },
  "test": {
    "rows_before": 3554,
    "rows_after": 2477,
    "removed_total": 1077,
    "removed_by_dedup": 215
  },
  "caps": {
    "train": {
      "split": "train",
      "original_rows": 1973,
      "kept_rows": 1973,
      "cap": 2000,
      "applied": false
    },
    "validation": {
      "split": "validation",
      "original_rows": 1311,
      "kept_rows": 800,
      "cap": 800,
      "applied": true
    },
    "test": {
      "split": "test",
      "original_rows": 2477,
      "kept_rows": 1200,
      "cap": 1200,
      "applied": true
    }
  },
  "cache_tag": "kaggle_fast_s2048_ctx1848_tr2000_va800_te1200_v2"
}
Final rows -> train / val / test: 1973 800 1200
Mean packed context tokens -> train / val / test: 

,question_text,answer_text,answer_type,packed_context_tokens,packed_context_used_chunks
0,What is the seed lexicon?,a vocabulary of positive and negative predicat...,abstractive,1848,1
1,What is the seed lexicon?,seed lexicon consists of positive and negative...,extractive,1848,1


## 5) Format prompts and tokenize

Prompts include the question and packed paper context, with the answer used as the supervised target.

In [7]:
SYSTEM_PROMPT = (
    "You are a scientific question-answering assistant. "
    "Use only the provided context. "
    "If the context is insufficient, output exactly INSUFFICIENT_CONTEXT."
)


EMPTY_CONTEXT_FALLBACK = "[CHUNK 1]\nNo reliable context is available for this sample."


def build_user_input(question: str, packed_context: str) -> str:
    q = normalize_text(question)
    c = normalize_text(packed_context)
    if not q:
        q = "What is the answer based on the provided context?"
    if not c:
        c = EMPTY_CONTEXT_FALLBACK

    return (
        "[CONTEXT]\n"
        f"{c}\n\n"
        "[QUESTION]\n"
        f"{q}\n\n"
        "[INSTRUCTION]\n"
        "Provide a concise, context-grounded answer."
    )


def to_instruction_df(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = df.copy()

    out["question_text"] = out["question_text"].fillna("").astype(str).map(normalize_text)
    out["packed_context"] = out["packed_context"].fillna("").astype(str).map(normalize_text)
    out["answer_text"] = out["answer_text"].fillna("").astype(str).map(normalize_text)

    out.loc[out["question_text"].str.len() == 0, "question_text"] = "What is the answer based on the provided context?"
    out.loc[out["packed_context"].str.len() == 0, "packed_context"] = EMPTY_CONTEXT_FALLBACK
    out.loc[out["answer_text"].str.len() == 0, "answer_text"] = "INSUFFICIENT_CONTEXT"

    out["instruction"] = SYSTEM_PROMPT
    out["input"] = out.apply(lambda r: build_user_input(r["question_text"], r["packed_context"]), axis=1)
    out["output"] = out["answer_text"].map(normalize_text)
    out["split"] = split_name

    # Drop any rows that still have an empty target.
    out.loc[out["output"].str.len() == 0, "output"] = "INSUFFICIENT_CONTEXT"
    return out


train_inst = to_instruction_df(train_df, "train")
val_inst = to_instruction_df(val_df, "validation")
test_inst = to_instruction_df(test_df, "test")

if ENABLE_CLASS_BALANCE_OVERSAMPLING:
    rare_train = train_inst[train_inst["answer_type"].isin(OVER_SAMPLE_TYPES)]
    if len(rare_train) > 0:
        train_inst = pd.concat([train_inst, rare_train], ignore_index=True)

# Shuffle after resampling.
train_inst = train_inst.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Mark difficult examples using the EDA heuristic.
# Low evidence recall also marks an example as difficult.
_ev_recall_col = train_inst.get("packed_context_evidence_recall")
_low_ev_recall = pd.Series(False, index=train_inst.index)
if _ev_recall_col is not None:
    _ev_valid = pd.to_numeric(_ev_recall_col, errors="coerce")
    _low_ev_recall = _ev_valid.notna() & (_ev_valid < 0.3)
train_inst["is_hard_sample"] = (
    (train_inst["answer_type"].isin(["hybrid", "abstractive"]))
    | (train_inst["packed_context_tokens"] > train_inst["packed_context_tokens"].quantile(0.95))
    | _low_ev_recall
).astype(int)

print("Instruction rows -> train / val / test:", len(train_inst), len(val_inst), len(test_inst))
print("Train hard-sample ratio:", round(train_inst["is_hard_sample"].mean(), 4))


def _fallback_chat_render(messages: List[Dict[str, str]], add_generation_prompt: bool) -> str:
    lines = []
    for m in messages:
        role = m["role"].strip().upper()
        lines.append(f"{role}:\n{m['content']}")
    if add_generation_prompt:
        lines.append("ASSISTANT:\n")
    return "\n\n".join(lines)


def build_chat_text(instruction: str, user_input: str, assistant_output: Optional[str] = None) -> str:
    msgs = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_input},
    ]
    if assistant_output is not None:
        msgs.append({"role": "assistant", "content": assistant_output})

    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=(assistant_output is None),
        )

    return _fallback_chat_render(msgs, add_generation_prompt=(assistant_output is None))


def tokenize_supervised_batch(batch: Dict[str, List[Any]]) -> Dict[str, List[List[int]]]:
    batch_input_ids: List[List[int]] = []
    batch_attention_mask: List[List[int]] = []
    batch_labels: List[List[int]] = []

    instructions = batch["instruction"]
    inputs = batch["input"]
    outputs = batch["output"]

    for instruction, user_input, output in zip(instructions, inputs, outputs):
        prompt_text = build_chat_text(instruction, user_input, None)
        full_text = build_chat_text(instruction, user_input, output)

        prompt_tokens = tokenizer(
            prompt_text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )
        full_tokens = tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )

        input_ids = full_tokens["input_ids"]
        attention_mask = full_tokens["attention_mask"]

        labels = input_ids.copy()
        prompt_len = min(len(prompt_tokens["input_ids"]), max(0, len(labels) - 1))
        if prompt_len > 0:
            labels[:prompt_len] = [-100] * prompt_len

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)

    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": batch_labels,
    }


@dataclass
class DynamicCausalCollator:
    tokenizer: Any
    pad_to_multiple_of: int = 8

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {
                "input_ids": f["input_ids"],
                "attention_mask": f["attention_mask"],
            }
            for f in features
        ]
        batch = self.tokenizer.pad(
            input_features,
            padding=True,
            return_tensors="pt",
            pad_to_multiple_of=self.pad_to_multiple_of,
        )

        max_len = batch["input_ids"].shape[1]
        labels = torch.full((len(features), max_len), -100, dtype=torch.long)
        for i, f in enumerate(features):
            cur = torch.tensor(f["labels"], dtype=torch.long)
            labels[i, : len(cur)] = cur

        batch["labels"] = labels
        return batch


def has_supervised_signal(labels: List[int]) -> bool:
    return any(x != -100 for x in labels)


train_ds = Dataset.from_pandas(train_inst, preserve_index=False)
val_ds = Dataset.from_pandas(val_inst, preserve_index=False)

TOKENIZE_BATCH_SIZE = 64
TOKENIZE_NUM_PROC = 1 if Path("/kaggle").exists() else max(1, min(4, (os.cpu_count() or 1)))

train_tok = train_ds.map(
    tokenize_supervised_batch,
    batched=True,
    batch_size=TOKENIZE_BATCH_SIZE,
    num_proc=TOKENIZE_NUM_PROC,
    remove_columns=train_ds.column_names,
    load_from_cache_file=ENABLE_CACHE,
)
val_tok = val_ds.map(
    tokenize_supervised_batch,
    batched=True,
    batch_size=TOKENIZE_BATCH_SIZE,
    num_proc=TOKENIZE_NUM_PROC,
    remove_columns=val_ds.column_names,
    load_from_cache_file=ENABLE_CACHE,
)

train_tok = train_tok.filter(lambda x: has_supervised_signal(x["labels"]))
val_tok = val_tok.filter(lambda x: has_supervised_signal(x["labels"]))

data_collator = DynamicCausalCollator(tokenizer=tokenizer)

print("Tokenized rows -> train / val:", len(train_tok), len(val_tok))
print("Mean sequence length (train):", round(float(np.mean([len(x["input_ids"]) for x in train_tok])), 2))

Instruction rows -> train / val / test: 1973 800 1200
Train hard-sample ratio: 0.3147


Map (num_proc=1):   0%|          | 0/1973 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1973 [00:00<?, ? examples/s]

Filter:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenized rows -> train / val: 1973 800
Mean sequence length (train): 1959.35


## 6) Load the model and attach QLoRA adapters

The model is quantized across the available GPUs before the LoRA adapters are attached.

In [8]:
# Reduce allocator fragmentation before loading the model.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

if NUM_GPUS >= 2:
    # balanced_low_0 leaves fewer layers on GPU 0 to make room for
    # training activations + gradients that accumulate on GPU 0.
    # GPU 0 = 10GiB for layers (reserve ~5GiB for activations)
    # GPU 1 = 14GiB for layers (take the bulk of the model)
    DEVICE_MAP_CANDIDATES = ["balanced_low_0", "balanced"]
    MAX_MEMORY = {0: "7GiB", 1: "14GiB", "cpu": "48GiB"}
elif NUM_GPUS == 1:
    DEVICE_MAP_CANDIDATES = ["auto"]
    MAX_MEMORY = {0: "13GiB", "cpu": "64GiB"}
else:
    DEVICE_MAP_CANDIDATES = ["auto"]
    MAX_MEMORY = None

print("Detected GPUs:", NUM_GPUS)
print("Device map candidates:", DEVICE_MAP_CANDIDATES)
if MAX_MEMORY is not None:
    print("Max memory:", MAX_MEMORY)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)


def get_used_cuda_devices(model_obj: torch.nn.Module) -> List[int]:
    used_cuda_devices: List[int] = []
    if hasattr(model_obj, "hf_device_map") and isinstance(model_obj.hf_device_map, dict):
        for _, dev in model_obj.hf_device_map.items():
            if isinstance(dev, int):
                used_cuda_devices.append(int(dev))
            elif isinstance(dev, str) and dev.startswith("cuda:"):
                try:
                    used_cuda_devices.append(int(dev.split(":")[-1]))
                except Exception:
                    pass
    return sorted(set(used_cuda_devices))


def load_base_model(device_map_value: str) -> torch.nn.Module:
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map=device_map_value,
        max_memory=MAX_MEMORY,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
        trust_remote_code=True,
    )


print("Loading quantized base model:", MODEL_ID)
model = None
selected_device_map = None

for dm in DEVICE_MAP_CANDIDATES:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    candidate = load_base_model(dm)
    used_devices = get_used_cuda_devices(candidate)

    if NUM_GPUS < 2 or len(used_devices) >= 2:
        model = candidate
        selected_device_map = dm
        break

    print(f"Device map '{dm}' used only GPUs {used_devices}; trying next candidate.")
    del candidate

if model is None:
    selected_device_map = DEVICE_MAP_CANDIDATES[0]
    model = load_base_model(selected_device_map)

print("Selected device map:", selected_device_map)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model.config, "pretraining_tp"):
    model.config.pretraining_tp = 1

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})


def prepare_model_for_qlora_low_mem(model_obj: torch.nn.Module) -> torch.nn.Module:
    for p in model_obj.parameters():
        p.requires_grad = False

    if hasattr(model_obj, "enable_input_require_grads"):
        model_obj.enable_input_require_grads()
    else:
        def _make_inputs_require_grad(module, inputs, output):
            del module, inputs
            output.requires_grad_(True)
        model_obj.get_input_embeddings().register_forward_hook(_make_inputs_require_grad)

    for name, param in model_obj.named_parameters():
        if ("norm" in name.lower()) and (param.dtype in (torch.float16, torch.bfloat16)):
            param.data = param.data.to(torch.float32)

    return model_obj


try:
    model = prepare_model_for_kbit_training(model)
except torch.cuda.OutOfMemoryError:
    print("prepare_model_for_kbit_training hit OOM; using low-memory fallback.")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    model = prepare_model_for_qlora_low_mem(model)

if hasattr(model, "hf_device_map") and isinstance(model.hf_device_map, dict):
    device_usage: Dict[str, int] = {}
    used_cuda_devices: List[int] = []
    for _, dev in model.hf_device_map.items():
        dev_key = f"cuda:{dev}" if isinstance(dev, int) else str(dev)
        device_usage[dev_key] = device_usage.get(dev_key, 0) + 1

        if isinstance(dev, int):
            used_cuda_devices.append(int(dev))
        elif isinstance(dev, str) and dev.startswith("cuda:"):
            try:
                used_cuda_devices.append(int(dev.split(":")[-1]))
            except Exception:
                pass

    used_cuda_devices = sorted(set(used_cuda_devices))
    print("Layer distribution:", device_usage)
    if NUM_GPUS >= 2:
        if len(used_cuda_devices) >= 2:
            print("Model is sharded across both GPUs:", used_cuda_devices)
        else:
            print("WARNING: model is not sharded across both GPUs. Used CUDA devices:", used_cuda_devices)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)


def mark_model_parallel_for_trainer(model_obj: torch.nn.Module) -> None:
    for candidate in [model_obj, getattr(model_obj, "model", None), getattr(model_obj, "base_model", None)]:
        if candidate is not None:
            setattr(candidate, "is_parallelizable", True)
            setattr(candidate, "model_parallel", True)


if NUM_GPUS >= 2:
    mark_model_parallel_for_trainer(model)
    print("Enabled model-parallel flags for Trainer multi-GPU compatibility.")

model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


Detected GPUs: 2
Device map candidates: ['balanced_low_0', 'balanced']
Max memory: {0: '7GiB', 1: '14GiB', 'cpu': '48GiB'}
Loading quantized base model: meta-llama/Llama-3.2-3B-Instruct


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device map 'balanced_low_0' used only GPUs []; trying next candidate.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Selected device map: balanced
Layer distribution: {'cuda:0': 6, 'cuda:1': 26}
Model is sharded across both GPUs: [0, 1]
Enabled model-parallel flags for Trainer multi-GPU compatibility.
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 7) Trainer setup and training

In [9]:
# Avoid clearing the CUDA cache through Trainer on affected T4 sessions.
# The original cache-clear call can fail with:
# AcceleratorError: CUDA error: unspecified launch failure
import gc
import transformers.trainer as _hf_trainer

try:
    import accelerate.utils.memory as _acc_memory
except Exception:
    _acc_memory = None


def _safe_clear_device_cache(*args, **kwargs):
    # Only run Python garbage collection here.
    gc.collect()


_hf_trainer.clear_device_cache = _safe_clear_device_cache
if _acc_memory is not None:
    _acc_memory.clear_device_cache = _safe_clear_device_cache

print("Applied safe clear_device_cache patch for Trainer/Accelerate.")

Applied safe clear_device_cache patch for Trainer/Accelerate.


In [10]:
bf16_supported = False
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability(0)
    bf16_supported = major >= 8

if len(train_tok) == 0:
    raise ValueError("train_tok is empty after preprocessing. Relax filtering settings.")
if len(val_tok) == 0:
    raise ValueError("val_tok is empty after preprocessing. Relax filtering settings.")


def is_cuda_oom_error(exc: BaseException) -> bool:
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True
    msg = str(exc).lower()
    return ("cuda" in msg) and ("out of memory" in msg)


def try_finite_float(value: Any) -> float:
    try:
        v = float(value)
    except Exception:
        return float("nan")
    if not np.isfinite(v):
        return float("nan")
    return float(v)


def finite_float(value: Any, default: float = 0.0) -> float:
    v = try_finite_float(value)
    return float(default) if np.isnan(v) else float(v)


# Cap optimizer updates to fit within a Kaggle session.
if RUNTIME_PROFILE == "kaggle_fast":
    max_updates_fast = 500
    current_updates = math.ceil(len(train_tok) / max(1, TRAIN_BSZ * GRAD_ACC_STEPS))
    if current_updates > max_updates_fast:
        target_samples = max_updates_fast * TRAIN_BSZ * GRAD_ACC_STEPS
        target_samples = min(target_samples, len(train_tok))
        train_tok = train_tok.shuffle(seed=SEED).select(range(target_samples))
        print(f"Applied runtime cap: train_tok reduced to {len(train_tok)} rows (~{max_updates_fast} updates max).")


def build_training_args() -> TrainingArguments:
    eval_strategy_value = TRAIN_EVAL_STRATEGY
    if eval_strategy_value not in {"no", "steps", "epoch"}:
        eval_strategy_value = "no"

    save_strategy_value = TRAIN_SAVE_STRATEGY
    if save_strategy_value not in {"steps", "epoch"}:
        save_strategy_value = "epoch"

    use_eval = eval_strategy_value != "no"

    # Loading the best checkpoint requires matching save and evaluation strategies.
    if LOAD_BEST_MODEL_AT_END and use_eval and save_strategy_value != eval_strategy_value:
        print(
            "Adjusting save strategy to match eval strategy for load_best_model_at_end: "
            f"{save_strategy_value} -> {eval_strategy_value}"
        )
        save_strategy_value = eval_strategy_value

    training_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=TRAIN_BSZ,
        per_device_eval_batch_size=EVAL_BSZ,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        num_train_epochs=EPOCHS,
        max_steps=-1,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=0.3,  # fixed at 0.3 for this run
        weight_decay=WEIGHT_DECAY,
        optim="paged_adamw_8bit",
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        save_strategy=save_strategy_value,
        save_total_limit=SAVE_TOTAL_LIMIT,
        fp16=not bf16_supported,
        bf16=bf16_supported,
        gradient_checkpointing=True,
        dataloader_num_workers=0 if Path("/kaggle").exists() else 2,
        dataloader_pin_memory=True,
        report_to="none",
        group_by_length=GROUP_BY_LENGTH,
        disable_tqdm=bool(Path("/kaggle").exists()),
        load_best_model_at_end=(LOAD_BEST_MODEL_AT_END and use_eval),
        do_eval=use_eval,
    )

    if use_eval:
        if "evaluation_strategy" in TrainingArguments.__dataclass_fields__:
            training_kwargs["evaluation_strategy"] = eval_strategy_value
        else:
            training_kwargs["eval_strategy"] = eval_strategy_value

        if eval_strategy_value == "steps":
            training_kwargs["eval_steps"] = EVAL_STEPS

        training_kwargs["metric_for_best_model"] = "eval_loss"
        training_kwargs["greater_is_better"] = False
    else:
        if "evaluation_strategy" in TrainingArguments.__dataclass_fields__:
            training_kwargs["evaluation_strategy"] = "no"
        else:
            training_kwargs["eval_strategy"] = "no"

    if save_strategy_value == "steps":
        training_kwargs["save_steps"] = SAVE_STEPS

    if "gradient_checkpointing_kwargs" in TrainingArguments.__dataclass_fields__:
        training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    if "torch_empty_cache_steps" in TrainingArguments.__dataclass_fields__:
        training_kwargs["torch_empty_cache_steps"] = 1

    args = TrainingArguments(**training_kwargs)

    # A sharded model must not be wrapped in DataParallel again.
    if NUM_GPUS >= 2:
        args._n_gpu = 1

    return args


def build_trainer(train_dataset: Dataset, eval_dataset: Dataset) -> Trainer:
    training_args = build_training_args()
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset if TRAIN_EVAL_STRATEGY != "no" else None,
        data_collator=data_collator,
    )

    try:
        trainer_obj = Trainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer_obj = Trainer(tokenizer=tokenizer, **trainer_kwargs)

    # Avoid a nested notebook entry point.
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer_obj.remove_callback(NotebookProgressCallback)
    except Exception:
        pass

    return trainer_obj


def retokenize_for_max_seq_length(new_max_seq_length: int) -> None:
    global MAX_SEQ_LENGTH, MAX_CONTEXT_TOKENS_IN_PROMPT, train_tok, val_tok
    MAX_SEQ_LENGTH = int(new_max_seq_length)
    # Keep the prompt budget in sync with the fallback sequence length.
    MAX_CONTEXT_TOKENS_IN_PROMPT = max(384, min(4096, MAX_SEQ_LENGTH - 200))
    print(f"Retokenizing datasets with MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}, "
          f"MAX_CONTEXT_TOKENS_IN_PROMPT={MAX_CONTEXT_TOKENS_IN_PROMPT} ...")

    train_tok = train_ds.map(
        tokenize_supervised_batch,
        batched=True,
        batch_size=64,
        num_proc=1 if Path("/kaggle").exists() else max(1, min(4, (os.cpu_count() or 1))),
        remove_columns=train_ds.column_names,
        load_from_cache_file=ENABLE_CACHE,
    )
    val_tok = val_ds.map(
        tokenize_supervised_batch,
        batched=True,
        batch_size=64,
        num_proc=1 if Path("/kaggle").exists() else max(1, min(4, (os.cpu_count() or 1))),
        remove_columns=val_ds.column_names,
        load_from_cache_file=ENABLE_CACHE,
    )

    train_tok = train_tok.filter(lambda x: has_supervised_signal(x["labels"]))
    val_tok = val_tok.filter(lambda x: has_supervised_signal(x["labels"]))

    if len(train_tok) == 0 or len(val_tok) == 0:
        raise ValueError("Tokenized datasets became empty after sequence-length fallback.")

    if RUNTIME_PROFILE == "kaggle_fast":
        max_updates_fast = 500
        current_updates = math.ceil(len(train_tok) / max(1, TRAIN_BSZ * GRAD_ACC_STEPS))
        if current_updates > max_updates_fast:
            target_samples = max_updates_fast * TRAIN_BSZ * GRAD_ACC_STEPS
            target_samples = min(target_samples, len(train_tok))
            train_tok = train_tok.shuffle(seed=SEED).select(range(target_samples))

    print("Tokenized rows -> train / val:", len(train_tok), len(val_tok))


def evaluate_with_fallback(trainer_obj: Trainer, eval_dataset: Dataset) -> Dict[str, float]:
    try:
        return trainer_obj.evaluate(eval_dataset=eval_dataset)
    except RuntimeError as exc:
        if "on_train_begin must be called before on_evaluate" not in str(exc):
            raise
        try:
            from transformers.utils.notebook import NotebookProgressCallback
            trainer_obj.remove_callback(NotebookProgressCallback)
        except Exception:
            pass
        return trainer_obj.evaluate(eval_dataset=eval_dataset)


def make_display_friendly_log(log_df: pd.DataFrame) -> pd.DataFrame:
    """Show training log with per-step loss and no empty/NaN values."""
    if log_df.empty:
        return log_df

    out = log_df.copy()
    out = out.dropna(axis=1, how="all")

    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            cleaned = pd.to_numeric(out[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            out[col] = cleaned.map(lambda v: "—" if pd.isna(v) else round(float(v), 6))
        else:
            out[col] = out[col].fillna("—")

    return out


base_max_seq_length = int(MAX_SEQ_LENGTH)
seq_candidates: List[int] = []
for candidate in [base_max_seq_length] + list(OOM_SEQ_LENGTH_FALLBACKS):
    try:
        c = int(candidate)
    except Exception:
        continue
    if c <= 0:
        continue
    if c > base_max_seq_length:
        continue
    if c not in seq_candidates:
        seq_candidates.append(c)

print("Training sequence-length candidates:", seq_candidates)

trainer = None
train_result = None
last_oom: Optional[Exception] = None

for seq_len in seq_candidates:
    if seq_len != MAX_SEQ_LENGTH:
        retokenize_for_max_seq_length(seq_len)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    gc.collect()

    approx_updates = math.ceil(len(train_tok) / max(1, TRAIN_BSZ * GRAD_ACC_STEPS))
    print(
        f"Planned training with MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}, "
        f"train_rows={len(train_tok)}, approx_updates={approx_updates}, "
        f"train_bs={TRAIN_BSZ}, grad_acc={GRAD_ACC_STEPS}, eval_strategy={TRAIN_EVAL_STRATEGY}."
    )

    trainer = None

    try:
        trainer = build_trainer(train_tok, val_tok)
        print("Starting training...")
        train_result = trainer.train()
        print("Training finished.")
        last_oom = None
        break
    except Exception as exc:
        if not is_cuda_oom_error(exc):
            raise
        last_oom = exc
        print(f"OOM at MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. Trying a smaller sequence length...")

        if trainer is not None:
            try:
                del trainer
            except Exception:
                pass

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
        gc.collect()
        continue

if train_result is None and TRAIN_BSZ > 1:
    # If sequence fallback is exhausted, reduce the batch size once.
    print(f"All seq-length candidates OOM with TRAIN_BSZ={TRAIN_BSZ}. "
          f"Falling back to TRAIN_BSZ=max(1, TRAIN_BSZ//2)...")
    TRAIN_BSZ = 1
    GRAD_ACC_STEPS = max(1, BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
    last_seq = seq_candidates[-1] if seq_candidates else 512
    retokenize_for_max_seq_length(last_seq)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    try:
        trainer = build_trainer(train_tok, val_tok)
        print(f"Retry training: TRAIN_BSZ={TRAIN_BSZ}, GRAD_ACC={GRAD_ACC_STEPS}, SEQ={MAX_SEQ_LENGTH}")
        train_result = trainer.train()
        print("Training finished after batch-size fallback.")
    except Exception as exc:
        if not is_cuda_oom_error(exc):
            raise
        last_oom = exc

if train_result is None:
    raise RuntimeError(
        "Training failed with CUDA OOM for all configured candidates. "
        "Lower MAX_SEQ_LENGTH to 512 and BATCH_SIZE to 1."
    ) from last_oom


# Run one validation pass after training.
final_eval_metrics = evaluate_with_fallback(trainer, val_tok)

trainer.save_state()
trainer.save_model(str(OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "adapter"))

log_history = pd.DataFrame(trainer.state.log_history)
log_history.to_csv(OUTPUT_DIR / "training_logs.csv", index=False)


# Keep the logged training loss by step.
if not log_history.empty and "loss" in log_history.columns:
    loss_by_step_df = log_history.copy()
    if "step" not in loss_by_step_df.columns:
        loss_by_step_df["step"] = np.arange(1, len(loss_by_step_df) + 1)

    keep_cols = [c for c in ["step", "epoch", "loss", "learning_rate"] if c in loss_by_step_df.columns]
    loss_by_step_df = loss_by_step_df[keep_cols].copy()
    loss_by_step_df["step"] = pd.to_numeric(loss_by_step_df["step"], errors="coerce")
    loss_by_step_df["loss"] = pd.to_numeric(loss_by_step_df["loss"], errors="coerce")
    if "learning_rate" in loss_by_step_df.columns:
        loss_by_step_df["learning_rate"] = pd.to_numeric(loss_by_step_df["learning_rate"], errors="coerce")
    if "epoch" in loss_by_step_df.columns:
        loss_by_step_df["epoch"] = pd.to_numeric(loss_by_step_df["epoch"], errors="coerce")

    loss_by_step_df = loss_by_step_df.replace([np.inf, -np.inf], np.nan)
    loss_by_step_df = loss_by_step_df.dropna(subset=["step", "loss"]).sort_values("step").reset_index(drop=True)
else:
    loss_by_step_df = pd.DataFrame(columns=["step", "epoch", "loss", "learning_rate"])

loss_by_step_df.to_csv(OUTPUT_DIR / "loss_per_step.csv", index=False)


def latest_finite_from_series(df: pd.DataFrame, col: str) -> float:
    if df.empty or col not in df.columns:
        return float("nan")
    s = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return float("nan")
    return float(s.iloc[-1])


training_loss = try_finite_float(getattr(train_result, "training_loss", np.nan))
if np.isnan(training_loss):
    training_loss = try_finite_float(train_result.metrics.get("train_loss", np.nan))
if np.isnan(training_loss):
    training_loss = latest_finite_from_series(log_history, "loss")
if np.isnan(training_loss):
    training_loss = finite_float(loss_by_step_df["loss"].mean(), default=0.0) if not loss_by_step_df.empty else 0.0

validation_loss = try_finite_float(final_eval_metrics.get("eval_loss", np.nan))
if np.isnan(validation_loss):
    validation_loss = latest_finite_from_series(log_history, "eval_loss")
if np.isnan(validation_loss):
    validation_loss = training_loss


def safe_exp(x: float) -> float:
    if x is None or np.isnan(x) or np.isinf(x):
        return float("nan")
    return float(math.exp(min(20.0, float(x))))


def estimate_chars_per_token_from_outputs(df: pd.DataFrame, sample_cap: int = 2048) -> float:
    if "output" not in df.columns or len(df) == 0:
        return 4.0

    sample = df["output"].astype(str).head(sample_cap).tolist()
    total_chars = 0
    total_tokens = 0
    for txt in sample:
        norm = normalize_text(txt)
        if not norm:
            continue
        total_chars += len(norm)
        total_tokens += len(tokenizer.encode(norm, add_special_tokens=False))

    if total_tokens <= 0:
        return 4.0
    return float(max(1.0, total_chars / total_tokens))


def loss_to_bpc(loss_value: float, chars_per_token: float) -> float:
    if not np.isfinite(loss_value):
        return 0.0
    bits_per_token = float(loss_value / math.log(2))
    bpc_value = bits_per_token / max(chars_per_token, 1e-6)
    return float(max(0.0, bpc_value))


chars_per_token_train = estimate_chars_per_token_from_outputs(train_inst)
chars_per_token_val = estimate_chars_per_token_from_outputs(val_inst)

optimizer_updates = math.ceil(len(train_tok) / max(1, TRAIN_BSZ * GRAD_ACC_STEPS))

best_logged_train_loss = finite_float(loss_by_step_df["loss"].min(), default=training_loss) if not loss_by_step_df.empty else float(training_loss)
last_logged_train_loss = finite_float(loss_by_step_df["loss"].iloc[-1], default=training_loss) if not loss_by_step_df.empty else float(training_loss)

auto_runtime_metrics = {
    "optimizer_updates": int(optimizer_updates),
    "effective_batch_size": int(TRAIN_BSZ * GRAD_ACC_STEPS),
    "max_seq_length_used": int(MAX_SEQ_LENGTH),
    "runtime_profile": RUNTIME_PROFILE,
    "loss_points_logged": int(len(loss_by_step_df)),
    "best_logged_train_loss": float(best_logged_train_loss),
    "final_logged_train_loss": float(last_logged_train_loss),
}

training_summary_metrics = {
    "training_loss": float(training_loss),
    "validation_loss": float(validation_loss),
    "training_perplexity": safe_exp(training_loss),
    "validation_perplexity": safe_exp(validation_loss),
    "training_bpc": loss_to_bpc(training_loss, chars_per_token_train),
    "validation_bpc": loss_to_bpc(validation_loss, chars_per_token_val),
    "train_chars_per_token": float(chars_per_token_train),
    "validation_chars_per_token": float(chars_per_token_val),
    "train_runtime_sec": finite_float(train_result.metrics.get("train_runtime", 0.0), default=0.0),
    "train_samples_per_second": finite_float(train_result.metrics.get("train_samples_per_second", 0.0), default=0.0),
    "train_steps_per_second": finite_float(train_result.metrics.get("train_steps_per_second", 0.0), default=0.0),
    **auto_runtime_metrics,
}


print("Loss per logging step (last rows):")
display(make_display_friendly_log(loss_by_step_df.tail(30)))

log_history_display = make_display_friendly_log(log_history.tail(20))
display(log_history_display)
print(json.dumps(training_summary_metrics, indent=2))

Training sequence-length candidates: [2048, 1536, 1024]


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Planned training with MAX_SEQ_LENGTH=2048, train_rows=1973, approx_updates=124, train_bs=1, grad_acc=16, eval_strategy=epoch.
Starting training...
{'loss': '1.54', 'grad_norm': '2.029', 'learning_rate': '1.739e-05', 'epoch': '0.04055'}
{'loss': '1.498', 'grad_norm': '2.403', 'learning_rate': '3.913e-05', 'epoch': '0.08109'}
{'loss': '1.495', 'grad_norm': '1.706', 'learning_rate': '6.087e-05', 'epoch': '0.1216'}
{'loss': '1.396', 'grad_norm': '2.376', 'learning_rate': '8.261e-05', 'epoch': '0.1622'}
{'loss': '1.434', 'grad_norm': '3.296', 'learning_rate': '0.0001', 'epoch': '0.2027'}
{'loss': '1.2', 'grad_norm': '3.036', 'learning_rate': '9.993e-05', 'epoch': '0.2433'}
{'loss': '1.252', 'grad_norm': '1.176', 'learning_rate': '9.976e-05', 'epoch': '0.2838'}
{'loss': '1.223', 'grad_norm': '1.237', 'learning_rate': '9.948e-05', 'epoch': '0.3244'}
{'loss': '1.442', 'grad_norm': '1.703', 'learning_rate': '9.911e-05', 'epoch': '0.3649'}
{'loss': '1.247', 'grad_norm': '1.589', 'learning_rate':

,step,epoch,loss,learning_rate
44,225.0,1.819057,1.060899,0.000038
45,230.0,1.859605,1.116000,0.000036
46,235.0,1.900152,0.916596,0.000034
47,240.0,1.940699,0.912827,0.000032
48,245.0,1.981247,0.966916,0.000030
49,250.0,2.016219,0.790023,0.000028
50,255.0,2.056766,1.109703,0.000026
51,260.0,2.097314,0.815364,0.000024
52,265.0,2.137861,0.718496,0.000022
53,270.0,2.178409,0.672554,0.000020


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
59,0.859132,2.170551,0.000013,2.340598,290.0,—,—,—,—,—,—,—,—,—
60,0.898892,1.816484,0.000012,2.381145,295.0,—,—,—,—,—,—,—,—,—
61,0.641692,1.737344,0.00001,2.421693,300.0,—,—,—,—,—,—,—,—,—
62,0.514802,2.907672,0.000009,2.462240,305.0,—,—,—,—,—,—,—,—,—
63,0.775179,0.946683,0.000008,2.502788,310.0,—,—,—,—,—,—,—,—,—
64,1.056792,1.490976,0.000007,2.543335,315.0,—,—,—,—,—,—,—,—,—
65,0.859616,1.204183,0.000006,2.583882,320.0,—,—,—,—,—,—,—,—,—
66,0.928303,1.734525,0.000005,2.624430,325.0,—,—,—,—,—,—,—,—,—
67,0.676998,1.925619,0.000004,2.664977,330.0,—,—,—,—,—,—,—,—,—
68,0.525224,2.311206,0.000003,2.705525,335.0,—,—,—,—,—,—,—,—,—


{
  "training_loss": 1.0082429998343991,
  "validation_loss": 1.4883917570114136,
  "training_perplexity": 2.7407812290011817,
  "validation_perplexity": 4.429965326323552,
  "training_bpc": 0.32669146941792737,
  "validation_bpc": 0.48681589532996206,
  "train_chars_per_token": 4.4524798228241655,
  "validation_chars_per_token": 4.410898303320495,
  "train_runtime_sec": 30410.9108,
  "train_samples_per_second": 0.195,
  "train_steps_per_second": 0.012,
  "optimizer_updates": 124,
  "effective_batch_size": 16,
  "max_seq_length_used": 2048,
  "runtime_profile": "kaggle_fast",
  "loss_points_logged": 74,
  "best_logged_train_loss": 0.5148015975952148,
  "final_logged_train_loss": 0.5357705593109131
}


## 8) Generation evaluation

Evaluation runs on the validation and test splits and keeps one record per example.

In [11]:
import logging
import warnings

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

for _logger_name in ["huggingface_hub.utils._headers", "huggingface_hub.file_download"]:
    logging.getLogger(_logger_name).setLevel(logging.ERROR)

warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module="evaluate")
warnings.filterwarnings("ignore", category=UserWarning, module="bert_score")

print("Loading evaluation metrics...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    bleu_metric = evaluate.load("sacrebleu")
    rouge_metric = evaluate.load("rouge")
    bertscore_metric = evaluate.load("bertscore") if ENABLE_BERTSCORE else None

DEFAULT_BERTSCORE_MODELS = [
    "distilbert-base-uncased",
    "roberta-base",
    "microsoft/deberta-v3-base",
]
if isinstance(globals().get("BERTSCORE_MODEL", None), str) and BERTSCORE_MODEL.strip():
    BERTSCORE_MODEL_CANDIDATES = [BERTSCORE_MODEL] + [m for m in DEFAULT_BERTSCORE_MODELS if m != BERTSCORE_MODEL]
else:
    BERTSCORE_MODEL_CANDIDATES = DEFAULT_BERTSCORE_MODELS


def resolve_eval_cap(split_name: str) -> Optional[int]:
    name = normalize_text(split_name).lower()
    if name.startswith("val"):
        return EVAL_MAX_SAMPLES_VALIDATION
    if name.startswith("test"):
        return EVAL_MAX_SAMPLES_TEST
    return None


def get_inference_device(model_obj: torch.nn.Module) -> torch.device:
    if hasattr(model_obj, "hf_device_map") and isinstance(model_obj.hf_device_map, dict):
        dev_values = list(model_obj.hf_device_map.values())
        for dev in dev_values:
            if isinstance(dev, torch.device):
                return dev
            if isinstance(dev, int):
                return torch.device(f"cuda:{dev}")
            if isinstance(dev, str) and dev.startswith("cuda"):
                return torch.device(dev)
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")


INFER_DEVICE = get_inference_device(model)
print("Inference device:", INFER_DEVICE)


# Cache attention states during generation.
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
model.config.use_cache = True

# Batched generation for a decoder-only model uses left padding.
# Training still uses right padding through the data collator;
# left padding keeps each prompt next to the generated continuation.
#
tokenizer.padding_side = 'left'
print('Tokenizer padding_side set to LEFT for correct decoder-only generation.')


def force_inference_dtype_compat(model_obj: torch.nn.Module) -> torch.dtype:
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        target_dtype = torch.float16 if major < 8 else torch.bfloat16
    else:
        target_dtype = torch.float32

    try:
        if hasattr(model_obj, "config") and hasattr(model_obj.config, "torch_dtype"):
            model_obj.config.torch_dtype = target_dtype
    except Exception:
        pass

    output_head = model_obj.get_output_embeddings() if hasattr(model_obj, "get_output_embeddings") else None
    if output_head is not None:
        if hasattr(output_head, "weight") and torch.is_floating_point(output_head.weight):
            output_head.weight.data = output_head.weight.data.to(target_dtype)
        if hasattr(output_head, "bias") and output_head.bias is not None and torch.is_floating_point(output_head.bias):
            output_head.bias.data = output_head.bias.data.to(target_dtype)

    return target_dtype


def patch_generate_with_dtype_retry(model_obj: torch.nn.Module) -> None:
    if getattr(model_obj, "_dtype_safe_generate_patched", False):
        return

    original_generate = model_obj.generate

    def _safe_generate(*args, **kwargs):
        use_autocast = torch.cuda.is_available()
        autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_autocast else nullcontext()
        try:
            with autocast_ctx:
                return original_generate(*args, **kwargs)
        except RuntimeError as exc:
            msg = str(exc)
            dtype_mismatch = "expected scalar type" in msg and any(t in msg for t in ["BFloat16", "Half", "Float"])
            if not dtype_mismatch:
                raise
            _ = force_inference_dtype_compat(model_obj)
            retry_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_autocast else nullcontext()
            with retry_ctx:
                return original_generate(*args, **kwargs)

    model_obj.generate = _safe_generate
    model_obj._dtype_safe_generate_patched = True


_ = force_inference_dtype_compat(model)
patch_generate_with_dtype_retry(model)


def normalize_answer_text(text: str) -> str:
    text = normalize_text(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def lexical_overlap_ratio(pred: str, context: str) -> float:
    pred_tokens = set(normalize_answer_text(pred).split())
    if len(pred_tokens) == 0:
        return 0.0
    ctx_tokens = set(normalize_answer_text(context).split())
    return float(len(pred_tokens & ctx_tokens) / max(1, len(pred_tokens)))


def token_prf_single(pred: str, gold: str) -> Tuple[float, float, float]:
    p_tokens = normalize_answer_text(pred).split()
    g_tokens = normalize_answer_text(gold).split()

    if len(p_tokens) == 0 and len(g_tokens) == 0:
        return 1.0, 1.0, 1.0
    if len(p_tokens) == 0 and len(g_tokens) > 0:
        return 0.0, 0.0, 0.0
    if len(g_tokens) == 0 and len(p_tokens) > 0:
        return 0.0, 0.0, 0.0

    p_count: Dict[str, int] = {}
    for t in p_tokens:
        p_count[t] = p_count.get(t, 0) + 1

    g_count: Dict[str, int] = {}
    for t in g_tokens:
        g_count[t] = g_count.get(t, 0) + 1

    common = 0
    for t in p_count:
        if t in g_count:
            common += min(p_count[t], g_count[t])

    if common == 0:
        return 0.0, 0.0, 0.0

    precision = common / len(p_tokens)
    recall = common / len(g_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return float(precision), float(recall), float(f1)


def token_accuracy_single(pred: str, gold: str) -> float:
    p_tokens = normalize_answer_text(pred).split()
    g_tokens = normalize_answer_text(gold).split()

    if len(p_tokens) == 0 and len(g_tokens) == 0:
        return 1.0

    max_len = max(len(p_tokens), len(g_tokens))
    if max_len == 0:
        return 1.0

    matches = 0
    for i in range(max_len):
        p_tok = p_tokens[i] if i < len(p_tokens) else None
        g_tok = g_tokens[i] if i < len(g_tokens) else None
        if p_tok is not None and g_tok is not None and p_tok == g_tok:
            matches += 1

    return float(matches / max_len)


def token_f1_single(pred: str, gold: str) -> float:
    return token_prf_single(pred, gold)[2]


def is_insufficient_context_answer(text: str) -> bool:
    raw = normalize_text(text).lower()
    norm = normalize_answer_text(text)

    cues = [
        "insufficient_context",
        "insufficient context",
        "not enough context",
        "cannot be determined",
        "cannot determine",
        "not provided in the context",
        "unknown based on context",
    ]
    return any(c in raw for c in cues) or any(c in norm for c in cues)


def extract_answer(raw_generation: str) -> str:
    text = normalize_text(raw_generation)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"^\s*(assistant|response)\s*:\s*", "", text, flags=re.IGNORECASE)

    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    cleaned_lines: List[str] = []

    for ln in lines:
        ln = re.sub(r"^\[(?:chunk|context)\s*\d+\]\s*", "", ln, flags=re.IGNORECASE)
        ln = re.sub(r"^\[(?:chunk|context)\]\s*", "", ln, flags=re.IGNORECASE)
        ln = re.sub(r"^(answer)\s*[:\-]\s*", "", ln, flags=re.IGNORECASE)
        ln = re.sub(r"^(according to|based on)\s+(the\s+)?(provided\s+)?context\s*[:,]?\s*", "", ln, flags=re.IGNORECASE)
        ln = re.sub(r"\((see|from)\s+chunk\s*\d+\)", "", ln, flags=re.IGNORECASE)
        ln = ln.strip(" -:\t")

        if not ln:
            continue
        if re.fullmatch(r"\[(?:chunk|context)\s*\d+\]", ln, flags=re.IGNORECASE):
            continue

        cleaned_lines.append(ln)

    if not cleaned_lines:
        return "INSUFFICIENT_CONTEXT"

    answer = " ".join(cleaned_lines[:4]).strip()
    answer = re.sub(r"\s+", " ", answer).strip()

    # Keep complete multi-sentence answers.

    max_words = int(max(24, min(140, int(globals().get("eda_answer_budget", 75)) + 24)))
    words = answer.split()
    if len(words) > max_words:
        answer = " ".join(words[:max_words]).strip()

    if not answer:
        return "INSUFFICIENT_CONTEXT"
    return answer


def expected_calibration_error(confidences: np.ndarray, correctness: np.ndarray, n_bins: int = 10) -> float:
    if confidences.size == 0:
        return 0.0

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= left) & (confidences <= right)
        else:
            mask = (confidences >= left) & (confidences < right)

        if not np.any(mask):
            continue

        bin_conf = float(np.mean(confidences[mask]))
        bin_acc = float(np.mean(correctness[mask]))
        ece += (np.sum(mask) / len(confidences)) * abs(bin_acc - bin_conf)

    return float(ece)


def proxy_confidence_from_text(pred_text: str, context_text: str) -> float:
    pred_norm = normalize_answer_text(pred_text)
    if not pred_norm:
        return float(CONFIDENCE_PROXY_FLOOR)

    if is_insufficient_context_answer(pred_text):
        return 0.62

    pred_tokens = set(pred_norm.split())
    ctx_tokens = set(normalize_answer_text(context_text).split())
    overlap = len(pred_tokens & ctx_tokens) / max(1, len(pred_tokens))

    length_penalty = 1.0 if len(pred_tokens) >= 3 else 0.8
    conf = (0.12 + 0.83 * overlap) * length_penalty
    conf = float(np.clip(conf, CONFIDENCE_PROXY_FLOOR, 0.99))
    return conf


def count_non_special_generated_tokens(token_ids: np.ndarray, special_ids: set) -> int:
    return int(sum(1 for tid in token_ids if int(tid) not in special_ids))


def generate_predictions(
    eval_df: pd.DataFrame,
    split_name: str,
    batch_size: int,
    max_new_tokens: int,
) -> Tuple[List[str], List[float], List[float], List[int], bool]:
    work = eval_df.copy().reset_index(drop=True)
    split_cap = resolve_eval_cap(split_name)
    if split_cap is not None:
        work = work.head(split_cap).copy()

    prompts = [build_chat_text(r["instruction"], r["input"], None) for _, r in work.iterrows()]

    preds: List[str] = []
    confidences: List[float] = []
    mean_logprobs: List[float] = []
    generated_token_counts: List[int] = []

    want_logprob = bool(ENABLE_TOKEN_LOGPROB_CONFIDENCE and CONFIDENCE_MODE in {"auto", "logprob"})

    special_ids = set(tokenizer.all_special_ids if tokenizer.all_special_ids is not None else [])
    if tokenizer.pad_token_id is not None:
        special_ids.add(int(tokenizer.pad_token_id))
    if tokenizer.eos_token_id is not None:
        special_ids.add(int(tokenizer.eos_token_id))

    adaptive_max_new_tokens = int(max(max_new_tokens, min(128, max(48, int(globals().get("eda_answer_budget", 75)) + 24))))

    model.eval()

    start = 0
    while start < len(prompts):
        current_bs = min(batch_size, len(prompts) - start)
        generated = None
        tokenized = None

        output_scores_flag = bool(want_logprob)

        while current_bs >= 1:
            batch_prompts = prompts[start:start + current_bs]
            tokenized = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            )
            tokenized = {k: v.to(INFER_DEVICE) for k, v in tokenized.items()}

            try:
                with torch.no_grad():
                    generated = model.generate(
                        **tokenized,
                        max_new_tokens=adaptive_max_new_tokens,
                        do_sample=False,
                        num_beams=1,
                        repetition_penalty=1.03,
                        no_repeat_ngram_size=4,
                        renormalize_logits=True,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        return_dict_in_generate=True,
                        output_scores=output_scores_flag,
                    )
                break
            except RuntimeError as exc:
                if not is_cuda_oom_error(exc):
                    raise

                if current_bs > 1:
                    current_bs = max(1, current_bs // 2)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        try:
                            torch.cuda.ipc_collect()
                        except Exception:
                            pass
                    gc.collect()
                    continue

                if output_scores_flag:
                    output_scores_flag = False
                    want_logprob = False
                    print("Falling back to proxy confidence because logprob generation hit OOM at batch_size=1.")
                    continue

                raise

        if generated is None or tokenized is None:
            raise RuntimeError("Generation failed for current batch.")

        input_len = tokenized["input_ids"].shape[1]
        gen_ids = generated.sequences[:, input_len:]
        decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        preds.extend([extract_answer(t) for t in decoded])

        gen_ids_np = gen_ids.detach().cpu().numpy()

        if output_scores_flag and generated.scores is not None and len(generated.scores) > 0:
            transition_scores = model.compute_transition_scores(
                generated.sequences,
                generated.scores,
                normalize_logits=True,
            )

            score_np = transition_scores.detach().cpu().numpy()

            for i in range(gen_ids_np.shape[0]):
                token_ids = gen_ids_np[i]
                token_scores = score_np[i]
                valid_logps: List[float] = []
                for j, tid in enumerate(token_ids):
                    if j >= len(token_scores):
                        break
                    if int(tid) in special_ids:
                        continue
                    lp = float(token_scores[j])
                    if np.isfinite(lp):
                        valid_logps.append(lp)

                mean_lp = float(np.mean(valid_logps)) if len(valid_logps) else -20.0
                conf = float(np.exp(np.clip(mean_lp, -20.0, 0.0)))
                token_count = len(valid_logps)
                if token_count <= 0:
                    token_count = count_non_special_generated_tokens(token_ids, special_ids)

                mean_logprobs.append(mean_lp)
                confidences.append(conf)
                generated_token_counts.append(int(max(1, token_count)))
        else:
            for i in range(gen_ids_np.shape[0]):
                token_ids = gen_ids_np[i]
                token_count = count_non_special_generated_tokens(token_ids, special_ids)
                mean_logprobs.append(float("nan"))
                confidences.append(float("nan"))
                generated_token_counts.append(int(max(1, token_count)))

        start += current_bs

    return preds, confidences, mean_logprobs, generated_token_counts, bool(want_logprob)


def safe_mean(values: List[float]) -> float:
    if len(values) == 0:
        return 0.0
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(arr.mean()) if arr.size else 0.0


def compute_text_metrics(predictions: List[str], references: List[str]) -> Dict[str, Any]:
    if len(predictions) == 0:
        return {
            "bleu": 0.0,
            "rouge_l": 0.0,
            "bertscore_precision": 0.0,
            "bertscore_recall": 0.0,
            "bertscore_f1": 0.0,
            "bertscore_status": "empty_input",
            "bertscore_model_used": "token_overlap_proxy",
        }

    try:
        bleu_out = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
        bleu_val = float(bleu_out.get("score", 0.0))
    except Exception as exc:
        print("BLEU computation failed:", exc)
        bleu_val = 0.0

    try:
        rouge_out = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)
        rouge_l_val = float(rouge_out.get("rougeL", 0.0))
    except Exception as exc:
        print("ROUGE computation failed:", exc)
        rouge_l_val = 0.0

    token_precisions: List[float] = []
    token_recalls: List[float] = []
    token_f1s: List[float] = []
    for p, r in zip(predictions, references):
        pp, rr, ff = token_prf_single(p, r)
        token_precisions.append(pp)
        token_recalls.append(rr)
        token_f1s.append(ff)

    fallback_p = safe_mean(token_precisions)
    fallback_r = safe_mean(token_recalls)
    fallback_f1 = safe_mean(token_f1s)

    bert_p, bert_r, bert_f1 = fallback_p, fallback_r, fallback_f1
    bert_status = "fallback_token_overlap"
    bert_model_used = "token_overlap_proxy"

    if ENABLE_BERTSCORE and bertscore_metric is not None:
        if BERTSCORE_MAX_SAMPLES is not None and len(predictions) > BERTSCORE_MAX_SAMPLES:
            rng = np.random.default_rng(SEED)
            idx = rng.choice(np.arange(len(predictions)), size=BERTSCORE_MAX_SAMPLES, replace=False)
            pred_subset = [predictions[int(i)] for i in idx]
            ref_subset = [references[int(i)] for i in idx]
        else:
            pred_subset = predictions
            ref_subset = references

        bert_errors: List[str] = []
        for model_name in BERTSCORE_MODEL_CANDIDATES:
            try:
                bert_out = bertscore_metric.compute(
                    predictions=pred_subset,
                    references=ref_subset,
                    lang="en",
                    model_type=model_name,
                    device=BERTSCORE_DEVICE,
                    batch_size=16,
                )
                bert_p = float(np.mean(bert_out["precision"]))
                bert_r = float(np.mean(bert_out["recall"]))
                bert_f1 = float(np.mean(bert_out["f1"]))
                bert_status = "ok"
                bert_model_used = model_name
                break
            except Exception as exc:
                bert_errors.append(f"{model_name}: {exc}")

        if bert_status != "ok":
            print("BERTScore failed for all model candidates. Using token-overlap fallback.")
            if bert_errors:
                print("Last BERTScore error:", bert_errors[-1])

    return {
        "bleu": bleu_val,
        "rouge_l": rouge_l_val,
        "bertscore_precision": bert_p,
        "bertscore_recall": bert_r,
        "bertscore_f1": bert_f1,
        "bertscore_status": bert_status,
        "bertscore_model_used": bert_model_used,
    }


def evaluate_split(eval_df: pd.DataFrame, split_name: str, conf_threshold: Optional[float] = None) -> Tuple[Dict[str, float], pd.DataFrame, float]:
    preds, raw_confs, mean_logps, generated_token_counts, logprob_was_used = generate_predictions(
        eval_df,
        split_name=split_name,
        batch_size=GEN_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
    )

    work = eval_df.copy().reset_index(drop=True)
    split_cap = resolve_eval_cap(split_name)
    if split_cap is not None:
        work = work.head(split_cap).copy()

    work["generated_answer"] = preds[:len(work)]
    work["confidence_raw_logprob"] = np.asarray(raw_confs[:len(work)], dtype=float)
    work["mean_logprob"] = np.asarray(mean_logps[:len(work)], dtype=float)
    work["generated_token_count"] = np.asarray(generated_token_counts[:len(work)], dtype=int)

    proxy_conf = np.asarray(
        [proxy_confidence_from_text(p, c) for p, c in zip(work["generated_answer"].tolist(), work["packed_context"].tolist())],
        dtype=float,
    )

    raw_conf = work["confidence_raw_logprob"].to_numpy(dtype=float)
    raw_conf_finite = np.isfinite(raw_conf)

    if CONFIDENCE_MODE == "proxy":
        use_logprob = np.zeros_like(raw_conf_finite, dtype=bool)
    else:
        use_logprob = raw_conf_finite

    if CONFIDENCE_MODE == "logprob" and not np.any(use_logprob):
        print("Logprob-only confidence requested but unavailable. Falling back to proxy confidence.")

    conf = np.where(use_logprob, raw_conf, proxy_conf)
    conf = np.nan_to_num(conf, nan=0.5, posinf=1.0, neginf=0.0)
    conf = np.clip(conf, 0.0, 1.0)

    confidence_source = np.where(use_logprob, "logprob", "proxy")

    # Abstain only when confidence and context overlap are both very low.
    guarded_answers: List[str] = []
    for pred, ctx, c in zip(work["generated_answer"].tolist(), work["packed_context"].tolist(), conf.tolist()):
        if is_insufficient_context_answer(pred):
            guarded_answers.append("INSUFFICIENT_CONTEXT")
            continue

        overlap = lexical_overlap_ratio(pred, ctx)
        pred_len = len(normalize_answer_text(pred).split())
        if pred_len >= 3 and c < 0.10 and overlap < 0.12:
            guarded_answers.append("INSUFFICIENT_CONTEXT")
        else:
            guarded_answers.append(pred)

    work["generated_answer"] = guarded_answers
    work["confidence"] = conf
    work["confidence_proxy"] = proxy_conf
    work["confidence_source"] = confidence_source

    em: List[int] = []
    token_precisions: List[float] = []
    token_recalls: List[float] = []
    token_f1_vals: List[float] = []
    token_acc_vals: List[float] = []
    grounded: List[int] = []
    abstention_ok: List[int] = []

    for _, r in work.iterrows():
        pred = r["generated_answer"]
        gold = r["output"]

        pred_n = normalize_answer_text(pred)
        gold_n = normalize_answer_text(gold)

        corr = int(pred_n == gold_n)
        em.append(corr)

        p_prec, p_rec, p_f1 = token_prf_single(pred, gold)
        token_precisions.append(p_prec)
        token_recalls.append(p_rec)
        token_f1_vals.append(p_f1)
        token_acc_vals.append(token_accuracy_single(pred, gold))

        overlap = lexical_overlap_ratio(pred, r["packed_context"])
        grounded.append(int((pred_n in normalize_answer_text(r["packed_context"])) or (overlap >= 0.5)))

        if r["answer_type"] == "unanswerable":
            abstention_ok.append(int(is_insufficient_context_answer(pred)))

    correctness = np.asarray(em, dtype=int)

    if conf_threshold is None:
        conf_threshold = float(np.quantile(conf, CONFIDENCE_QUANTILE)) if conf.size else 0.5
    if not np.isfinite(conf_threshold):
        conf_threshold = 0.5

    confident_mask = conf >= conf_threshold
    confident_wrong_mask = confident_mask & (correctness == 0)

    try:
        brier = float(brier_score_loss(correctness, conf)) if conf.size else 0.0
    except Exception:
        brier = float(np.mean((correctness - conf) ** 2)) if conf.size else 0.0

    ece = expected_calibration_error(conf, correctness, n_bins=10)
    try:
        if len(np.unique(correctness)) > 1:
            conf_auc = float(roc_auc_score(correctness, conf))
        else:
            conf_auc = 0.5
    except Exception:
        conf_auc = 0.5

    avg_confidence = float(np.mean(conf)) if conf.size else 0.0
    confident_answer_pct = float(np.mean(confident_mask)) if conf.size else 0.0
    confident_wrong_overall_pct = float(np.mean(confident_wrong_mask)) if conf.size else 0.0
    critical_error_rate = float(np.sum(confident_wrong_mask) / max(1, np.sum(confident_mask)))

    token_count_arr = np.asarray(work["generated_token_count"].to_numpy(dtype=float), dtype=float)
    token_count_arr = np.clip(token_count_arr, 1.0, None)
    pred_char_arr = np.asarray([max(1, len(normalize_text(x))) for x in work["generated_answer"].tolist()], dtype=float)
    mean_lp_arr = work["mean_logprob"].to_numpy(dtype=float)

    valid_lp_mask = np.isfinite(mean_lp_arr) & np.isfinite(token_count_arr)
    if np.any(valid_lp_mask):
        total_bits = float(np.sum((-mean_lp_arr[valid_lp_mask] / math.log(2)) * token_count_arr[valid_lp_mask]))
        total_chars = float(np.sum(pred_char_arr[valid_lp_mask]))
        bpc = total_bits / max(1.0, total_chars)
        bpc_source = "logprob"
    else:
        pseudo_bits = -np.log2(np.clip(conf, 1e-6, 1.0))
        total_bits = float(np.sum(pseudo_bits * token_count_arr))
        total_chars = float(np.sum(pred_char_arr))
        bpc = total_bits / max(1.0, total_chars)
        bpc_source = "proxy_confidence"

    bpc = float(np.clip(bpc, 0.0, 64.0))

    txt_metrics = compute_text_metrics(
        predictions=work["generated_answer"].tolist(),
        references=work["output"].tolist(),
    )

    work["is_correct"] = [bool(x) for x in em]
    work["is_grounded"] = [bool(x) for x in grounded]

    n_unanswerable = len(abstention_ok)
    abstention_acc = float(np.mean(abstention_ok)) if n_unanswerable > 0 else 0.0

    metrics = {
        "split": split_name,
        "n_eval": int(len(work)),
        "exact_match": float(np.mean(em)) if em else 0.0,
        "token_precision": safe_mean(token_precisions),
        "token_recall": safe_mean(token_recalls),
        "token_f1": safe_mean(token_f1_vals),
        "token_accuracy": safe_mean(token_acc_vals),
        "bleu": txt_metrics["bleu"],
        "rouge_l": txt_metrics["rouge_l"],
        "bertscore_precision": txt_metrics["bertscore_precision"],
        "bertscore_recall": txt_metrics["bertscore_recall"],
        "bertscore_f1": txt_metrics["bertscore_f1"],
        "bertscore_status": txt_metrics["bertscore_status"],
        "bertscore_model_used": txt_metrics["bertscore_model_used"],
        "grounding_rate": float(np.mean(grounded)) if grounded else 0.0,
        "hallucination_rate": float(1.0 - np.mean(grounded)) if grounded else 0.0,
        "unanswerable_count": int(n_unanswerable),
        "unanswerable_abstention_accuracy": abstention_acc,
        "confidence_threshold": float(conf_threshold),
        "avg_confidence": avg_confidence,
        "confident_answer_pct": confident_answer_pct,
        "confident_wrong_overall_pct": confident_wrong_overall_pct,
        "critical_error_rate": critical_error_rate,
        "ece": ece,
        "brier_score": brier,
        "confidence_auc": conf_auc,
        "bpc": bpc,
        "bpc_source": bpc_source,
        "avg_generated_tokens": float(np.mean(token_count_arr)) if token_count_arr.size else 0.0,
        "avg_generated_chars": float(np.mean(pred_char_arr)) if pred_char_arr.size else 0.0,
        "confidence_available": True,
        "confidence_source_logprob_pct": float(np.mean(confidence_source == "logprob")),
        "confidence_source_proxy_pct": float(np.mean(confidence_source == "proxy")),
        "logprob_confidence_runtime_active": bool(logprob_was_used),
    }

    return metrics, work, float(conf_threshold)


print("Evaluating validation split...")
val_metrics, val_pred_df, global_conf_threshold = evaluate_split(val_inst, "validation", conf_threshold=None)
print("Evaluating test split...")
test_metrics, test_pred_df, _ = evaluate_split(test_inst, "test", conf_threshold=global_conf_threshold)

# Metrics by answer type.
try:
    for split_tag in ["validation", "test"]:
        split_inst = val_inst if split_tag == "validation" else test_inst
        if "answer_type" in split_inst.columns:
            type_counts = split_inst["answer_type"].value_counts().to_dict()
            print(f"Answer type distribution ({split_tag}): {type_counts}")
except Exception:
    pass

metrics_df = pd.DataFrame([val_metrics, test_metrics])
display(metrics_df)

Loading evaluation metrics...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inference device: cuda:0
Tokenizer padding_side set to LEFT for correct decoder-only generation.
Evaluating validation split...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating test split...
Answer type distribution (validation): {'extractive': 400, 'abstractive': 400}
Answer type distribution (test): {'abstractive': 600, 'extractive': 600}


,split,n_eval,exact_match,token_precision,token_recall,token_f1,token_accuracy,bleu,rouge_l,bertscore_precision,bertscore_recall,bertscore_f1,bertscore_status,bertscore_model_used,grounding_rate,hallucination_rate,unanswerable_count,unanswerable_abstention_accuracy,confidence_threshold,avg_confidence,confident_answer_pct,confident_wrong_overall_pct,critical_error_rate,ece,brier_score,confidence_auc,bpc,bpc_source,avg_generated_tokens,avg_generated_chars,confidence_available,confidence_source_logprob_pct,confidence_source_proxy_pct,logprob_confidence_runtime_active
0,validation,600,0.06000,0.479534,0.293130,0.314723,0.135839,3.440111,0.315378,0.820931,0.761865,0.787772,ok,distilbert-base-uncased,0.913333,0.086667,0,0.0,0.721202,0.583843,0.250,0.211667,0.846667,0.523843,0.348036,0.760195,0.19341,logprob,10.476667,46.408333,True,1.0,0.0,True
1,test,800,0.04625,0.448816,0.255489,0.281230,0.108848,3.139669,0.280448,0.817000,0.762652,0.786712,ok,distilbert-base-uncased,0.890000,0.110000,0,0.0,0.721202,0.585824,0.265,0.230000,0.867925,0.539574,0.354874,0.840548,0.21830,logprob,11.816250,47.615000,True,1.0,0.0,True


In [12]:
# Evidence-overlap metrics.
if EVAL_CONTEXT_RECALL:
    for split_name, split_inst_df in [("validation", val_inst), ("test", test_inst)]:
        if "packed_context_evidence_recall" in split_inst_df.columns:
            ev_recall_valid = split_inst_df["packed_context_evidence_recall"].dropna()
            if len(ev_recall_valid) > 0:
                print(f"Evidence recall in packed context ({split_name}): "
                      f"mean={ev_recall_valid.mean():.3f}, "
                      f"median={ev_recall_valid.median():.3f}")

metrics_df_clean = metrics_df.replace([np.inf, -np.inf], np.nan).fillna(0.0).copy()
for col in metrics_df_clean.columns:
    if pd.api.types.is_numeric_dtype(metrics_df_clean[col]):
        metrics_df_clean[col] = metrics_df_clean[col].map(lambda v: float(v))

print("Sanitized metrics table (no NaN/inf values):")
display(metrics_df_clean)

Evidence recall in packed context (validation): mean=0.880, median=1.000
Evidence recall in packed context (test): mean=0.879, median=1.000
Sanitized metrics table (no NaN/inf values):


,split,n_eval,exact_match,token_precision,token_recall,token_f1,token_accuracy,bleu,rouge_l,bertscore_precision,bertscore_recall,bertscore_f1,bertscore_status,bertscore_model_used,grounding_rate,hallucination_rate,unanswerable_count,unanswerable_abstention_accuracy,confidence_threshold,avg_confidence,confident_answer_pct,confident_wrong_overall_pct,critical_error_rate,ece,brier_score,confidence_auc,bpc,bpc_source,avg_generated_tokens,avg_generated_chars,confidence_available,confidence_source_logprob_pct,confidence_source_proxy_pct,logprob_confidence_runtime_active
0,validation,600.0,0.06000,0.479534,0.293130,0.314723,0.135839,3.440111,0.315378,0.820931,0.761865,0.787772,ok,distilbert-base-uncased,0.913333,0.086667,0.0,0.0,0.721202,0.583843,0.250,0.211667,0.846667,0.523843,0.348036,0.760195,0.19341,logprob,10.476667,46.408333,1.0,1.0,0.0,1.0
1,test,800.0,0.04625,0.448816,0.255489,0.281230,0.108848,3.139669,0.280448,0.817000,0.762652,0.786712,ok,distilbert-base-uncased,0.890000,0.110000,0.0,0.0,0.721202,0.585824,0.265,0.230000,0.867925,0.539574,0.354874,0.840548,0.21830,logprob,11.816250,47.615000,1.0,1.0,0.0,1.0


## 9) Save the evaluation files

In [13]:
def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, (np.floating,)):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def as_float(value: Any, default: float = 0.0) -> float:
    try:
        v = float(value)
    except Exception:
        return float(default)
    if math.isnan(v) or math.isinf(v):
        return float(default)
    return float(v)


def as_int(value: Any, default: int = 0) -> int:
    try:
        v = int(value)
    except Exception:
        return int(default)
    return int(v)


def as_non_empty_text(value: Any, default: str = "N/A") -> str:
    txt = normalize_text(value)
    return txt if txt else default


def count_nulls(obj: Any) -> int:
    if obj is None:
        return 1
    if isinstance(obj, dict):
        return sum(count_nulls(v) for v in obj.values())
    if isinstance(obj, list):
        return sum(count_nulls(v) for v in obj)
    return 0


def clean_training_curve_rows(log_df: pd.DataFrame) -> List[Dict[str, Any]]:
    if log_df.empty:
        return []

    keep_cols = [c for c in ["epoch", "step", "loss", "eval_loss", "learning_rate"] if c in log_df.columns]
    rows = log_df[keep_cols].to_dict(orient="records")

    cleaned_rows: List[Dict[str, Any]] = []
    for r in rows:
        cleaned: Dict[str, Any] = {}
        for k, v in r.items():
            jv = to_jsonable(v)
            if jv is not None:
                cleaned[k] = jv
        if cleaned:
            cleaned_rows.append(cleaned)

    return cleaned_rows


def build_loss_per_step_rows(df: pd.DataFrame) -> List[Dict[str, Any]]:
    if df is None or len(df) == 0:
        return []

    out = df.copy()
    keep_cols = [c for c in ["step", "epoch", "loss", "learning_rate"] if c in out.columns]
    if not keep_cols:
        return []

    out = out[keep_cols].copy()
    if "step" in out.columns:
        out["step"] = pd.to_numeric(out["step"], errors="coerce")
    if "epoch" in out.columns:
        out["epoch"] = pd.to_numeric(out["epoch"], errors="coerce")
    if "loss" in out.columns:
        out["loss"] = pd.to_numeric(out["loss"], errors="coerce")
    if "learning_rate" in out.columns:
        out["learning_rate"] = pd.to_numeric(out["learning_rate"], errors="coerce")

    out = out.replace([np.inf, -np.inf], np.nan)
    if "step" in out.columns and "loss" in out.columns:
        out = out.dropna(subset=["step", "loss"]).sort_values("step")

    rows: List[Dict[str, Any]] = []
    for _, r in out.iterrows():
        row: Dict[str, Any] = {
            "step": as_int(r.get("step", 0), default=0),
            "loss": as_float(r.get("loss", 0.0), default=0.0),
        }
        if "epoch" in out.columns:
            row["epoch"] = as_float(r.get("epoch", 0.0), default=0.0)
        if "learning_rate" in out.columns:
            row["learning_rate"] = as_float(r.get("learning_rate", 0.0), default=0.0)
        rows.append(row)

    return rows


training_curve = []
if "log_history" in globals() and isinstance(log_history, pd.DataFrame) and not log_history.empty:
    training_curve = clean_training_curve_rows(log_history)

loss_per_step_rows = []
if "loss_by_step_df" in globals() and isinstance(loss_by_step_df, pd.DataFrame):
    loss_per_step_rows = build_loss_per_step_rows(loss_by_step_df)

metrics_payload = {
    "artifact_schema_version": 2,
    "model_id": MODEL_ID,
    "config": {
        "RUNTIME_PROFILE": as_non_empty_text(RUNTIME_PROFILE, default="unknown"),
        "EPOCHS": as_int(EPOCHS, default=1),
        "BATCH_SIZE": as_int(BATCH_SIZE, default=1),
        "GRADIENT_ACCUMULATION_STEPS": as_int(GRADIENT_ACCUMULATION_STEPS, default=1),
        "LEARNING_RATE": as_float(LEARNING_RATE, default=1e-4),
        "MAX_SEQ_LENGTH": as_int(MAX_SEQ_LENGTH, default=1024),
        "chunk_size": as_int(CHUNK_SIZE, default=384),
        "chunk_stride": as_int(CHUNK_STRIDE, default=96),
        "max_context_tokens_in_prompt": as_int(MAX_CONTEXT_TOKENS_IN_PROMPT, default=1024),
        "max_train_samples": as_int(MAX_TRAIN_SAMPLES, default=-1),
        "max_val_samples": as_int(MAX_VAL_SAMPLES, default=-1),
        "max_test_samples": as_int(MAX_TEST_SAMPLES, default=-1),
        "eval_max_samples_validation": as_int(EVAL_MAX_SAMPLES_VALIDATION, default=-1),
        "eval_max_samples_test": as_int(EVAL_MAX_SAMPLES_TEST, default=-1),
        "train_eval_strategy": as_non_empty_text(TRAIN_EVAL_STRATEGY, default="no"),
        "train_save_strategy": as_non_empty_text(TRAIN_SAVE_STRATEGY, default="epoch"),
        "confidence_mode": as_non_empty_text(CONFIDENCE_MODE, default="auto"),
        "enable_token_logprob_confidence": bool(ENABLE_TOKEN_LOGPROB_CONFIDENCE),
        "confidence_proxy_floor": as_float(CONFIDENCE_PROXY_FLOOR, default=0.05),
        "enable_bertscore": bool(ENABLE_BERTSCORE),
        "bertscore_max_samples": as_int(BERTSCORE_MAX_SAMPLES, default=-1),
        "bertscore_device": as_non_empty_text(BERTSCORE_DEVICE, default="cpu"),
        "use_train_val_resplit": bool(USE_TRAIN_VAL_RESPLIT),
        "train_val_resplit_ratio": as_float(TRAIN_VAL_RESPLIT_RATIO, default=0.9),
        "train_val_resplit_stratify": bool(TRAIN_VAL_RESPLIT_STRATIFY),
    },
    "dataset_stats": {
        "train_rows_after_preprocessing": int(len(train_df)),
        "validation_rows_after_preprocessing": int(len(val_df)),
        "test_rows_after_preprocessing": int(len(test_df)),
    },
    "evidence_analysis": {
        "evidence_columns_available": bool("evidence_text" in train_df.columns),
        "use_evidence_for_packing": USE_EVIDENCE_FOR_PACKING,
        "evidence_boost_weight": EVIDENCE_BOOST_WEIGHT,
        "train_evidence_recall_mean": float(
            train_df["packed_context_evidence_recall"].dropna().mean()
            if "packed_context_evidence_recall" in train_df.columns
            and len(train_df["packed_context_evidence_recall"].dropna()) > 0
            else 0.0
        ),
        "val_evidence_recall_mean": float(
            val_df["packed_context_evidence_recall"].dropna().mean()
            if "packed_context_evidence_recall" in val_df.columns
            and len(val_df["packed_context_evidence_recall"].dropna()) > 0
            else 0.0
        ),
        "eda_evidence_coverage_pct": float(eda_evidence_coverage) if "eda_evidence_coverage" in dir() else 0.0,
    },
    "training_summary": training_summary_metrics,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "training_curve": training_curve,
    "loss_per_step": loss_per_step_rows,
}


def build_eval_samples(df: pd.DataFrame, split_name: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    if df is None or len(df) == 0:
        return rows

    for _, r in df.iterrows():
        rows.append(
            {
                "split": as_non_empty_text(split_name, default="unknown"),
                "question": as_non_empty_text(r.get("question_text", ""), default="N/A"),
                "generated_answer": as_non_empty_text(r.get("generated_answer", ""), default="INSUFFICIENT_CONTEXT"),
                "reference_answer": as_non_empty_text(r.get("output", ""), default="INSUFFICIENT_CONTEXT"),
                "confidence": as_float(r.get("confidence", 0.5), default=0.5),
                "confidence_source": as_non_empty_text(r.get("confidence_source", "proxy"), default="proxy"),
                "confidence_proxy": as_float(r.get("confidence_proxy", 0.5), default=0.5),
                "mean_logprob": as_float(r.get("mean_logprob", -20.0), default=-20.0),
                "answer_type": as_non_empty_text(r.get("answer_type", "unknown"), default="unknown"),
                "is_grounded": bool(r.get("is_grounded", False)) if pd.notna(r.get("is_grounded", False)) else False,
                "is_correct": bool(r.get("is_correct", False)) if pd.notna(r.get("is_correct", False)) else False,
                "paper_id": as_non_empty_text(r.get("paper_id", ""), default="N/A"),
                "question_uid": as_non_empty_text(r.get("question_uid", ""), default="N/A"),
                "supporting_context": as_non_empty_text(r.get("packed_context", ""), default="N/A")[:2500],
            }
        )
    return rows


validation_samples = build_eval_samples(val_pred_df, "validation")
test_samples = build_eval_samples(test_pred_df, "test")

evaluation_payload = {
    "artifact_schema_version": 2,
    "model_id": MODEL_ID,
    "confidence_threshold": as_float(global_conf_threshold, default=0.5),
    "validation": validation_samples,
    "test": test_samples,
}

metrics_jsonable = to_jsonable(metrics_payload)
evaluation_jsonable = to_jsonable(evaluation_payload)

metrics_nulls = count_nulls(metrics_jsonable)
evaluation_nulls = count_nulls(evaluation_jsonable)

if metrics_nulls > 0 or evaluation_nulls > 0:
    raise ValueError(
        f"Export validation failed: metrics nulls={metrics_nulls}, evaluation nulls={evaluation_nulls}. "
        "Check confidence and metric fallbacks before exporting artifacts."
    )

# Save the main files in the output directory.
metrics_out = OUTPUT_DIR / "metrics.json"
evaluation_out = OUTPUT_DIR / "evaluation.json"
with metrics_out.open("w", encoding="utf-8") as f:
    json.dump(metrics_jsonable, f, indent=2)
with evaluation_out.open("w", encoding="utf-8") as f:
    json.dump(evaluation_jsonable, f, indent=2)

# Also copy the required files to the working directory.
with Path("metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics_jsonable, f, indent=2)
with Path("evaluation.json").open("w", encoding="utf-8") as f:
    json.dump(evaluation_jsonable, f, indent=2)

# Save the remaining diagnostic artifacts.
val_pred_df.to_csv(PRED_DIR / "validation_predictions.csv", index=False)
test_pred_df.to_csv(PRED_DIR / "test_predictions.csv", index=False)
train_inst.to_parquet(OUTPUT_DIR / "train_instruction_dataset.parquet", index=False)
val_inst.to_parquet(OUTPUT_DIR / "validation_instruction_dataset.parquet", index=False)
test_inst.to_parquet(OUTPUT_DIR / "test_instruction_dataset.parquet", index=False)

print("Saved required files:")
print("-", metrics_out)
print("-", evaluation_out)
print("-", Path("metrics.json").resolve())
print("-", Path("evaluation.json").resolve())
print("Null counts (metrics, evaluation):", metrics_nulls, evaluation_nulls)
print("Loss-per-step points:", len(loss_per_step_rows))

Saved required files:
- /kaggle/working/qasper_llama_3_1_8b_qlora/metrics.json
- /kaggle/working/qasper_llama_3_1_8b_qlora/evaluation.json
- /kaggle/working/metrics.json
- /kaggle/working/evaluation.json
Null counts (metrics, evaluation): 0 0
Loss-per-step points: 74


## 10) Check the saved files

In [14]:
required_outputs = [
    Path("metrics.json"),
    Path("evaluation.json"),
    OUTPUT_DIR / "metrics.json",
    OUTPUT_DIR / "evaluation.json",
]

missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required outputs: {missing}")

summary_df = pd.DataFrame(
    [
        {
            "path": str(p),
            "exists": p.exists(),
            "size_bytes": int(p.stat().st_size),
        }
        for p in required_outputs
    ]
)
display(summary_df)
print("Notebook pipeline completed successfully.")

,path,exists,size_bytes
0,metrics.json,True,26820
1,evaluation.json,True,4503614
2,/kaggle/working/qasper_llama_3_1_8b_qlora/metr...,True,26820
3,/kaggle/working/qasper_llama_3_1_8b_qlora/eval...,True,4503614


Notebook pipeline completed successfully.
